# CS781 Phase 7 · Prior Case Retrieval · Kaggle T4 × 2

**Run all cells in order.** This notebook trains **seven candidates from your original Phase-1 checkpoints**, evaluates on official development relevance, and creates a checked submission ZIP immediately after each run. An optional rank-fusion ensemble provides one additional score candidate. `best_submission.zip` is selected by development **Micro-F1@7**.

The score-oriented design is **BM25 shortlist → trained hierarchical document scorer → joint query–case cross-encoder reranking**. It preserves a strong lexical signal while learning relevance from your own foundations. No externally pretrained language model, external sentence embeddings, LLM/API output, or Phase-2–6 adapted checkpoint is loaded.

**Measured quality and GPU runtime require execution on your Kaggle data.** The included synthetic CPU tests verify implementation behavior; they do not establish a leaderboard score. Phase-5 Macro-F1 and Phase-6 BERTScore are different metrics.

Sources checked November 7, 2026: [capstone brief, §13](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit), [official dataset](https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone), [official Phase-7 evaluator](https://drive.google.com/file/d/19Raf18n0ysaIsvaMOwtt56az7qpF-lOC/view). The brief lists a Phase-7 deadline of **November 7, 2026, 11:59 PM IST**; submissions are left to you. Seven trained candidates exceed its stricter general minimum of six, provided their predictions are distinct. The ensemble does not count as another adaptation technique.

Authorship: AI-assisted implementation draft. Research sources and implementation differences are documented below for attribution and review.

## Correction for your executed notebook

Run 01 passed the actual two-T4 preflight. Run 02 stopped at a check combining loss and gradient-norm finiteness; the old log did not distinguish nonfinite forward loss from ordinary FP16 gradient-scaling overflow. No training epochs ran in the uploaded notebook.

This revision starts the GradScaler at 1024, retries a preflight microbatch with a halved scale when its gradients overflow, and reports loss/gradient/scale diagnostics. Persistent invalid gradients and nonfinite forward losses still stop. All ranks skip an invalid optimizer step together. Training reuses the scale calibrated by its preflight. The update test checks a matrix with a real gradient, rather than a softmax bias that can legitimately have zero gradient. Rare float32 norm-reduction overflow is handled with a stable float64 reduction. No NaN replacement or loss clamping is used.

The dependency cell now preserves installed packages rather than downgrading tokenizers/Hugging Face Hub. The logged pip conflicts were separate from the preflight exception.

**Existing Kaggle session:** run the supplied `Phase7_Preflight_Repair_Cell.py` as one cell immediately before the failed preflight, then rerun the preflight and subsequent cells. It replaces only `p7_run.py` and leaves the existing token/BM25 cache and configuration intact. No kernel restart or data rebuild is needed for that repair. For a fresh session, run this corrected notebook in order. Historical outputs have been cleared so they are not mistaken for results of the corrected code.

The uploaded audit's development Micro-F1@7 **0.204558** is the BM25 baseline. Your learned models have not yet been evaluated. It also reports a top-128 shortlist oracle Micro-F1@7 ceiling of **0.493608** on development data; a higher development target would require improved/expanded retrieval coverage, not merely more reranker epochs. This numerical repair leaves that retrieval configuration unchanged.

CPU regression tests reproduce/recover FP16 gradient overflow, simulate peer-only overflow, reject genuine nonfinite losses, bound persistent bad-gradient retries, verify stable norm clipping, and run all seven model workflows. Actual behavior on your original T4 checkpoints must be confirmed by rerunning the preflight.

Reference: [PyTorch 2.11 AMP examples](https://docs.pytorch.org/docs/2.11/notes/amp_examples.html).


## Required inputs

Attach your **original Phase-1 outputs**. No outputs from Phases 2–6 are required.

```text
/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/
  tokenizer.json
  runs/main_decoder/best_model.pt
  runs/main_encoder_decoder/best_model.pt
  runs/main_local_gqa/best_model.pt
  runs/main_hybrid/best_model.pt

<any attached official dataset>/Phase_7_PCR/
  Train/train.json
  Train/train_candidates.json
  Dev/dev.json
  Dev/dev_candidates.json
  Test/test.json
  Test/test_candidates.json
```

The dataset can instead be downloaded automatically at a pinned Hugging Face commit. If it is gated, accept its terms and enable the Kaggle secret `HF_TOKEN`. Enable Internet for installation/download, select **GPU T4 × 2**, and keep tokens out of notebook source.

Official query records are mappings such as `{"0000021652": {"text": ["sentence..."], "relevant_candidates": ["0000590455"]}}`; candidate mappings contain text. List-of-record and column-oriented alternatives are supported. Original IDs stay strings, including leading zeros.

Only **training queries and training relevance** drive updates/mining/distillation. Development labels are for selection. Test labels, if supplied, are discarded. Each split uses its own candidate pool. Candidate overlaps across pools are legitimate shared precedents and are not deleted. Exact held-out query copies are removed from training with an audit.

In [ ]:
import os, sys, subprocess, importlib.util, importlib.metadata
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf_cache_phase7')
# Preserve Kaggle's installed dependency versions; install a package only if it is absent.
requirements = [('tokenizers', 'tokenizers>=0.19'),
                ('huggingface_hub', 'huggingface-hub>=0.24'),
                ('sklearn', 'scikit-learn>=1.3')]
missing = [req for module, req in requirements if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Required packages already installed; keeping current versions.')
for package in ('tokenizers', 'huggingface-hub', 'scikit-learn'):
    print(package, importlib.metadata.version(package))


In [ ]:
from pathlib import Path
import json, shutil, torch
PROJECT = Path('/kaggle/working/capstone_phase7_v1')
PHASE1_ROOT = Path('/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1')
DATA_ROOT = None       # Or '/kaggle/input/<dataset>/Phase_7_PCR'. None auto-detects or downloads.
DATA_REVISION = None   # Resolved once and recorded as the exact official dataset commit.
WORLD_SIZE = 2
SETTINGS = {
    'seed': 781,
    'chunk_tokens': 384, 'chunk_overlap': 64, 'max_chunks': 4,
    'lexical_topk': 128, 'bm25_features': 200000,
    'micro_batch': 2, 'accumulation': 4, 'negatives': 3,
    'chunk_microbatch': 4, 'index_batch': 8,
    'pair_tokens': 512, 'pair_microbatch': 4, 'inference_pairs': 8,
    'rerank_topk': 64, 'dev_probe': 128,
    'full_epochs': 4, 'peft_epochs': 4, 'cross_epochs': 4, 'student_epochs': 3,
    'patience': 2, 'teacher_candidates': 16,
    'mine_hard_negatives': True, 'ensemble': True,
}
assert torch.cuda.is_available() and torch.cuda.device_count() >= WORLD_SIZE, 'Select GPU T4 × 2.'
assert SETTINGS['lexical_topk'] >= SETTINGS['rerank_topk'] >= 7
assert 0 <= SETTINGS['chunk_overlap'] < SETTINGS['chunk_tokens']
PROJECT.mkdir(parents=True, exist_ok=True)
CODE = PROJECT/'code'; CODE.mkdir(exist_ok=True)
print('PyTorch:', torch.__version__)
for i in range(WORLD_SIZE): print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/2**30, 1), 'GiB')
print('Effective query batch:', SETTINGS['micro_batch']*SETTINGS['accumulation']*WORLD_SIZE)
print('Free disk GiB:', round(shutil.disk_usage(PROJECT).free/2**30, 1))
# Set the configuration before starting; changed inputs/code/configuration require a new PROJECT.


## The seven trained candidates

| Run | Original Phase-1 family | Adaptation | Relevance model |
|---|---|---|---|
| 01 | Decoder Transformer | Full fine-tuning | Hierarchical shared bi-encoder |
| 02 | Encoder–decoder | Partial last-two decoder blocks | Encoder memory + four learned decoder readouts |
| 03 | Local GQA | LoRMA+ rank 16 | Chunk-level MaxSim + global similarity |
| 04 | LSTM → Transformer hybrid | 32 learned prompt embeddings | Hierarchical bi-encoder |
| 05 | Decoder Transformer | IA³ activation scaling | Hierarchical bi-encoder |
| 06 | Decoder Transformer | Full fine-tuning | Joint query–candidate cross-encoder |
| 07 | Smaller decoder | Supervised + retrieval KD | Half-depth bi-encoder student |

Every run restores its own original Phase-1 parent before adding task modules. Parent provenance, tokenizer hash, original architecture and 50M–300M size are checked. The smaller KD student is initialized by uniform layer selection from the original Phase-1 decoder, then learns from the strongest completed teacher; it does not inherit task-adapted teacher weights.

**Long documents:** all token IDs are cached once in a memory-mapped file. Each document encoder uses up to four selected overlapping windows: first, last, then windows with high training-corpus BPE-IDF salience. The neural view is a bounded subset, not full-document encoding. Token attention and position-aware chunk attention produce normalized 256-dimensional document vectors. The local-GQA family preserves its GQA projections but uses bidirectional full attention within each retrieval chunk. Decoder self-attention is adapted to bidirectional chunk encoding for retrieval; this is documented and is not presented as unchanged causal language modeling.

The native encoder–decoder encodes a chunk, then uses four learned decoder readouts with cross-attention to that memory. The hybrid keeps its original recurrent component; packed sequences prevent padding from changing recurrent state. The cross-encoder combines a compressed query with its candidate’s most query-matching token window, adds segment embeddings and lexical overlap features, and learns a joint relevance score. It does not see every candidate token.

**Scoring:** neural and lexical scores are standardized within a shortlist and combined using a small development-only weight grid. Seven model ZIPs retain a nonzero neural contribution. A sparse-only BM25 diagnostic is reported separately. The optional ensemble applies reciprocal-rank fusion to the best two/three learned rankers and is selected on development data.

## Training objective and research basis

For each query, known relevant candidates are excluded from negative sampling. Hard negatives come from BM25; after run 01 they are refreshed once using its trained scores. One random negative reduces dependence on a fixed hard list. All known positives represented in the batch are marked positive, including another query's selected positive if it is also relevant to this query.

With $P_q$ the positive columns present in a batch, the supervised loss is

$$L_{sup}=-\log\frac{\sum_{p\in P_q}\exp(s(q,p)/\tau)}{\sum_{d\in B}\exp(s(q,d)/\tau)},\quad \tau=0.05.$$

Candidate vectors are gathered **with gradient paths across the two ranks**, so each microbatch gets cross-GPU negatives. Gradient accumulation increases the optimizer batch; it does **not** make negatives span all accumulated microbatches. The cross-encoder instead uses grouped multi-positive listwise loss plus a binary relevance loss.

The student combines supervised loss with cached teacher distributions:

$$L=(1-\lambda)L_{sup}+\lambda T^2\,KL\left(softmax(z_T/T)\parallel softmax(z_S/T)\right).$$

Teacher targets are actual scores from the selected teacher's neural/lexical retriever on **training-only candidate groups**. Training positives may be included in those groups, but no gold candidates are injected into development/test shortlists. The student uses per-group cached KL, plus cross-rank supervised contrastive learning; it is not a full reproduction of TCT's in-batch KD.

Research read and applied:

- [Joshi et al., U-CREAT, ACL 2023](https://aclanthology.org/2023.acl-long.777/), experiments and legal retrieval analysis: supports keeping lexical retrieval strong on IL-PCR. This notebook uses BM25 and learned legal representations; it does not implement their event dependency parser, import their models, or claim their reported scores.
- [Karpukhin et al., DPR, EMNLP 2020, §3.2](https://aclanthology.org/2020.emnlp-main.550/): hard/in-batch negatives and precomputable representations. Here the encoder is your own shared foundation, with a multi-positive objective and BM25 shortlisting rather than a global FAISS dense search.
- [Santhanam et al., ColBERTv2, NAACL 2022, §3.1–3.2](https://aclanthology.org/2022.naacl-main.272/): fine-grained MaxSim and teacher-score supervision. Run 03 applies MaxSim to **chunk vectors**, with a global-score contribution; it is not token-level ColBERTv2, residual compression or PLAID.
- [Nogueira & Cho, Passage Re-ranking with BERT, §2](https://arxiv.org/abs/1901.04085): retrieve a shortlist and jointly encode query/candidate pairs for relevance. Run 06 uses your own decoder weights, segment embeddings and selected evidence; no BERT weights are imported.
- [Lin et al., RepL4NLP 2021](https://aclanthology.org/2021.repl4nlp-1.17/): ranking-specific distillation and the distinction between per-group and in-batch teacher supervision. We cache actual training-group scores once to avoid a live teacher during every student step.
- [Hofstätter et al., SIGIR 2021](https://arxiv.org/abs/2104.06967): efficient teacher supervision and negative sampling. Its resource results motivate avoiding repeated index refreshes; this notebook does not implement TAS clustering or claim its speedups.
- [Bihany et al., LoRMA, ACL Findings 2025, Eq. 8](https://aclanthology.org/2025.findings-acl.527/): $y=W_0x+(\alpha/r)BA(W_0x)$, identity-preserving initialization $B=0$. Rank-16 LoRMA+ adapts attention and feed-forward matrices.
- [Liu et al., IA³/T-Few, NeurIPS 2022](https://arxiv.org/abs/2205.05638): learned scaling vectors for keys, values and feed-forward intermediate activations. We implement those scales, not the T0 model or complete T-Few classification recipe.
- [Lester et al., Prompt Tuning, EMNLP 2021](https://aclanthology.org/2021.emnlp-main.243/): learned input prompts with a frozen backbone. Run 04 also learns its task-specific retrieval head; scale-dependent findings from large T5 models are not a promise for your smaller hybrid.

Runtime reductions: cached tokenization/BM25, bounded neural chunks, independent candidate encoding reused across queries in a pass, shortlist-only cross-encoding, one-time hard-negative mining, cached KD scores, FP16/GradScaler, checkpointed Transformer blocks, DDP accumulation with `no_sync`, early stopping and completed-run reuse. LoRMA/IA³ adapters are merged temporarily for indexing/ranking. The LSTM remains outside checkpoint recomputation. No BF16 or FlashAttention-2 requirement is imposed on T4.

## Runtime implementation

The following cells embed every runtime module, including the original Phase-1 model definitions and the unchanged official Phase-7 evaluator. No prior notebook imports are required.

In [ ]:
# evaluate_phase7_official.py
module_source = r'''#!/usr/bin/env python3
"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).

It uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /
Micro-Recall@7 definitions as the Codabench scorer.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

K = 7


def reject_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON numeric value '{value}' is not allowed.")


def pairs_no_duplicates(pairs):
    result = {}
    duplicates = []
    for key, value in pairs:
        if key in result:
            duplicates.append(key)
        result[key] = value
    if duplicates:
        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")
    return result


def load_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc


def load_truth(path: Path) -> dict[str, list[str]]:
    data = load_json(path)
    # Accept either the simple gold.json mapping or a Phase-7 dev query file:
    # {query_id: {text: [...], relevant_candidates: [...]}}
    if not isinstance(data, dict) or not data:
        raise ValueError("Truth must be a non-empty JSON object.")
    out: dict[str, list[str]] = {}
    for query_id, value in data.items():
        if isinstance(value, dict) and "relevant_candidates" in value:
            labels = value["relevant_candidates"]
        else:
            labels = value
        if not isinstance(query_id, str) or not query_id.strip():
            raise ValueError("Truth contains an empty/invalid query ID.")
        if not isinstance(labels, list):
            raise ValueError(f"Truth for query '{query_id}' must contain a list of relevant candidate IDs.")
        cleaned=[]; seen=set()
        for idx,candidate_id in enumerate(labels):
            if not isinstance(candidate_id,str) or not candidate_id.strip():
                # Treat blank source labels as absent labels, matching the builder.
                continue
            if candidate_id in seen:
                raise ValueError(f"Truth for query '{query_id}' contains duplicate candidate ID '{candidate_id}'.")
            seen.add(candidate_id); cleaned.append(candidate_id)
        out[query_id]=cleaned
    return out


def load_candidate_ids(path: Path) -> set[str]:
    data = load_json(path)
    if not isinstance(data, list) or not data:
        raise ValueError("candidate_ids must be a non-empty JSON list.")
    ids=[]; seen=set()
    for idx,v in enumerate(data):
        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")
        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID '{v}'.")
        seen.add(v); ids.append(v)
    return set(ids)


def load_prediction(path: Path) -> Any:
    return load_json(path)


def validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:
    errors=[]
    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]
    gold_ids=set(gold); pred_ids=set(pred)
    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID '{q}'.")
    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID '{q}'.")
    for q in sorted(gold_ids & pred_ids):
        vals=pred[q]
        if not isinstance(vals,list):
            errors.append(f"Query '{q}': prediction must be a list."); continue
        if len(vals)!=K: errors.append(f"Query '{q}': expected exactly {K} candidates, got {len(vals)}.")
        seen=set()
        for i,c in enumerate(vals):
            if not isinstance(c,str) or not c.strip(): errors.append(f"Query '{q}': index {i} is not a non-empty string."); continue
            if c in seen: errors.append(f"Query '{q}': duplicate candidate ID '{c}'.")
            seen.add(c)
            if c not in candidates: errors.append(f"Query '{q}': candidate ID '{c}' is outside the candidate pool.")
    return errors


def score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:
    tp=fp=fn=0
    for q,labels in gold.items():
        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}


def parse_args():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")
    p.add_argument("--prediction",type=Path,required=True)
    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")
    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")
    return p.parse_args()


def main():
    a=parse_args()
    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)
    errs=validate(pred,gold,candidates)
    if errs:
        print("LOCAL VALIDATION FAILED")
        for i,e in enumerate(errs,1): print(f"  {i}. {e}")
        print(f"Total validation errors: {len(errs)}")
        raise SystemExit(1)
    result=score(gold,pred)
    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}
    print("Validation passed.")
    print(f"Micro-F1@7: {scores['primary_metric_1']:.10f}")
    print(f"Micro-Precision@7: {scores['secondary_metric_1']:.10f}")
    print(f"Micro-Recall@7: {scores['secondary_metric_2']:.10f}")
    print(f"TP={result['tp']} FP={result['fp']} FN={result['fn']}")
    if a.output:
        a.output.parent.mkdir(parents=True,exist_ok=True)
        with a.output.open('w',encoding='utf-8') as f: json.dump(scores,f,indent=2); f.write('\n')
        print(f"Wrote {a.output}")

if __name__=='__main__': main()
'''
(CODE/'evaluate_phase7_official.py').write_text(module_source, encoding='utf-8')
print('Wrote evaluate_phase7_official.py')

In [ ]:
# legal_models.py
module_source = r'''"""Four randomly initialized legal language models; no downloaded model weights."""
import math
from dataclasses import dataclass, asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "decoder"
    vocab_size: int = 32768
    dim: int = 512
    heads: int = 8
    kv_heads: int = 8
    layers: int = 8
    ff_dim: int = 2048
    encoder_layers: int = 0
    rnn_layers: int = 0
    local_window: int = 0
    max_positions: int = 4096
    dropout: float = 0.0
    gradient_checkpointing: bool = False


def architecture_config(name, **overrides):
    c = ModelConfig(architecture=name)
    if name == "decoder":
        pass
    elif name == "local_gqa":
        c.kv_heads, c.layers, c.local_window = 2, 9, 256
    elif name == "hybrid":
        c.layers, c.rnn_layers = 7, 2
    elif name == "encoder_decoder":
        c.layers, c.encoder_layers = 4, 3
    else:
        raise ValueError(f"Unknown architecture: {name}")
    for k, v in overrides.items():
        if not hasattr(c, k):
            raise ValueError(k)
        setattr(c, k, v)
    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0
    assert (c.dim // c.heads) % 2 == 0
    return c


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Reduction in FP32 is useful with T4 FP16 training.
        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class Rotary(nn.Module):
    def __init__(self, head_dim, max_positions):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        angles = torch.outer(torch.arange(max_positions).float(), inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x, offset=0):
        n = x.size(-2)
        if offset + n > self.cos.size(0):
            raise ValueError("Position limit exceeded; increase max_positions in the model config.")
        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]
        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c, causal=True, cross=False):
        super().__init__()
        self.heads, self.kv_heads = c.heads, c.kv_heads
        self.hd = c.dim // c.heads
        self.q = nn.Linear(c.dim, c.dim, bias=False)
        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.out = nn.Linear(c.dim, c.dim, bias=False)
        self.rotary = None if cross else Rotary(self.hd, c.max_positions)
        self.causal, self.cross = causal, cross
        self.window = c.local_window if causal and not cross else 0
        self.dropout = c.dropout

    def forward(self, x, memory=None, past=None, position=0, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)
        if self.cross and past is not None:
            k, v = past  # Encoder K/V projections are also cached during decoding.
        else:
            source = x if memory is None else memory
            s = source.size(1)
            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            if self.rotary is not None:
                k = self.rotary(k, position)
            if past is not None:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rotary is not None:
            q = self.rotary(q, position)
        new_cache = (k, v) if use_cache else None
        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.
        # This is not a block-sparse implementation or a guaranteed training speedup.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k = k.repeat_interleave(repeats, dim=1)
            v = v.repeat_interleave(repeats, dim=1)
        mask, is_causal = None, False
        if self.causal:
            s = k.size(-2)
            if self.window or (past is not None and t > 1):
                qi = torch.arange(s - t, s, device=x.device)[:, None]
                ki = torch.arange(s, device=x.device)[None, :]
                mask = ki <= qi
                if self.window:
                    mask = mask & ((qi - ki) < self.window)
            elif past is None:
                is_causal = True
            # Cached single-token query must see all prior keys. is_causal=True
            # here would incorrectly use the upper-left non-square mask.
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))
        return y, new_cache


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, c, encoder=False):
        super().__init__()
        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)
        self.attn = Attention(c, causal=not encoder)
        self.ff = SwiGLU(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, past=None, position=0, use_cache=False):
        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache) if use_cache else x


class CrossBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))
        self.self_attn = Attention(c)
        self.cross_attn = Attention(c, causal=False, cross=True)
        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)

    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):
        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache, cross_cache) if use_cache else x


class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        # Output projection uses embedding.weight; there is no second vocabulary matrix.
        if c.architecture == "encoder_decoder":
            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))
            self.encoder_norm = RMSNorm(c.dim)
            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))
        else:
            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))
        if c.rnn_layers:
            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)
            self.rnn_norm = RMSNorm(c.dim)
        self.final_norm = RMSNorm(c.dim)
        self.apply(self._init)
        # Scale residual projections to keep initial residual variance controlled.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.normal_(p, mean=0.0, std=0.02)
                else:
                    nn.init.zeros_(p)

    def encode(self, source_ids):
        x = self.embedding(source_ids)
        for block in self.encoder:
            if self.config.gradient_checkpointing and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.encoder_norm(x)

    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):
        state = state or {}
        position = state.get("position", 0)
        x = self.embedding(input_ids)
        new_state = {"position": position + input_ids.size(1), "self": []}
        if self.config.rnn_layers:
            y, recurrent_state = self.rnn(x, state.get("rnn"))
            x = x + self.rnn_norm(y)
            new_state["rnn"] = recurrent_state
        if self.config.architecture == "encoder_decoder":
            memory = state.get("memory")
            if memory is None:
                if source_ids is None:
                    raise ValueError("encoder_decoder requires source_ids on the first call.")
                memory = self.encode(source_ids)
            new_state.update(memory=memory, cross=[])
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    cp = state.get("cross", [None] * len(self.blocks))[i]
                    x, cache, cc = block(x, memory, p, cp, position, True)
                    new_state["self"].append(cache)
                    new_state["cross"].append(cc)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, memory, use_reentrant=False)
                else:
                    x = block(x, memory)
        else:
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    x, cache = block(x, p, position, True)
                    new_state["self"].append(cache)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
        x = self.final_norm(x)
        return (x, new_state) if use_cache else x

    def project(self, hidden):
        return F.linear(hidden, self.embedding.weight)


def parameter_count(model):
    # PyTorch de-duplicates shared Parameter objects here.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def parameter_formula(c):
    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads
    attention = 2 * d * d + 2 * d * kv * h
    ff = 3 * d * c.ff_dim
    total = c.vocab_size * d + d
    if c.architecture == "encoder_decoder":
        total += c.encoder_layers * (attention + ff + 2 * d) + d
        total += c.layers * (2 * attention + ff + 3 * d)
    else:
        total += c.layers * (attention + ff + 2 * d)
    if c.rnn_layers:
        total += c.rnn_layers * (8 * d * d + 8 * d) + d
    return total


def make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):
    # blocks has T+1 tokens. The target suffix is identical across architectures.
    t = blocks.size(1) - 1
    split = max(1, min(t, int(t * prefix_fraction)))
    if architecture == "encoder_decoder":
        source = blocks[:, :split]
        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]
    else:
        source = None
        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()
        if objective == "continuation":
            targets[:, :split - 1] = -100
    return inputs, targets, source


def language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):
    """Sum/token count lets validation aggregate the true token-weighted NLL."""
    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)
    total = hidden.new_zeros((), dtype=torch.float32)
    count = (targets != -100).sum()
    def loss_chunk(h, y, w):
        # Keep logits in autocast precision until the bounded FP32 CE conversion.
        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")
    for i in range(0, targets.numel(), chunk_tokens):
        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]
        if checkpoint_chunks and torch.is_grad_enabled():
            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)
        else:
            total = total + loss_chunk(h, y, weight)
    return total / count.clamp_min(1), total.detach(), count.detach()


if __name__ == "__main__":
    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):
        c = architecture_config(name)
        with torch.device("meta"):
            model = LegalLM(c)
        n = parameter_count(model)
        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000
        print(name, f"{n:,}", asdict(c))
'''
(CODE/'legal_models.py').write_text(module_source, encoding='utf-8')
print('Wrote legal_models.py')

In [ ]:
# p7_data.py
module_source = r'''"""Official split-aware PCR data; IDs stay strings, test relevance never enters the cache."""
import collections,functools,hashlib,json,math,pickle,re
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

FILES={s:(f'{folder}/{s}.json',f'{folder}/{s}_candidates.json') for s,folder in [('train','Train'),('dev','Dev'),('test','Test')]}
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def read_json(path):
    def unique(pairs):
        d={}
        for k,v in pairs:
            if k in d:raise ValueError(f'Duplicate JSON mapping key {k!r}.')
            d[k]=v
        return d
    def bad(x):raise ValueError(f'Nonstandard JSON number {x}.')
    return json.loads(Path(path).read_text(encoding='utf-8-sig'),object_pairs_hook=unique,parse_constant=bad)
def atomic_json(path,x):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding='utf-8');tmp.replace(path)
def atomic_pickle(path,x):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    with tmp.open('wb') as f:pickle.dump(x,f,pickle.HIGHEST_PROTOCOL)
    tmp.replace(path)
def load_pickle(path):
    with open(path,'rb') as f:return pickle.load(f)
def alias(row,keys):
    present=[k for k in keys if k in row]
    if len(present)>1 and any(row[k]!=row[present[0]] for k in present):raise ValueError(f'Conflicting aliases: {present}')
    return row[present[0]] if present else None
def records(value,split,queries=True):
    if isinstance(value,dict) and len(value)==1 and next(iter(value)) in ('data','records','documents'):value=next(iter(value.values()))
    if isinstance(value,list):items=[(None,r) for r in value]
    elif isinstance(value,dict):
        ids=alias(value,('id','ids','case_id','doc_id'));texts=alias(value,('text','document','sentences'))
        gold=None if not queries or split=='test' else alias(value,('relevant_candidates','relevant','labels'))
        if isinstance(ids,str) and texts is not None:items=[(None,value)]
        elif isinstance(ids,list) and isinstance(texts,list):
            if len(ids)!=len(texts) or (queries and split!='test' and (not isinstance(gold,list) or len(gold)!=len(ids))):raise ValueError('Column lengths differ.')
            items=[(None,{'id':did,'text':texts[i],**({'relevant_candidates':gold[i]} if queries and split!='test' else {})}) for i,did in enumerate(ids)]
        elif isinstance(texts,dict):
            if queries and split!='test' and (not isinstance(gold,dict) or set(gold)!=set(texts)):raise ValueError('Column keys differ.')
            items=[(None,{'id':k,'text':t,**({'relevant_candidates':gold[k]} if queries and split!='test' else {})}) for k,t in texts.items()]
        else:items=list(value.items())
    else:raise ValueError('Expected an ID mapping or list of record objects.')
    out={};duplicates=0;blank_labels=0
    for key,row in items:
        if key is not None and isinstance(row,(str,list)):row={'id':key,'text':row}
        if not isinstance(row,dict):raise ValueError(f'{split}: record must be an object, got {type(row).__name__}.')
        did=alias(row,('id','case_id','doc_id'));did=key if did is None else did
        if not isinstance(did,str) or not did.strip():raise ValueError('Original nonempty string IDs are required, including leading zeros.')
        if key is not None and did!=key:raise ValueError('Mapping key and explicit document ID differ.')
        text=alias(row,('text','document','sentences'))
        if isinstance(text,list) and all(isinstance(s,str) for s in text):text='\n'.join(text)
        if not isinstance(text,str) or not text.strip():raise ValueError(f'{split}/{did}: expected nonempty text or sentence strings.')
        rel=None
        if queries and split!='test':
            rel=alias(row,('relevant_candidates','relevant','labels'))
            if not isinstance(rel,list):raise ValueError(f'{split}/{did}: relevant_candidates must be a list.')
            clean=[];seen=set()
            for c in rel:
                # Same blank-source policy as official load_truth; no integer coercion or zero-padding guesses.
                if not isinstance(c,str) or not c.strip():blank_labels+=1;continue
                if c in seen:raise ValueError(f'{split}/{did}: duplicate relevance ID {c}.')
                seen.add(c);clean.append(c)
            rel=clean
        # Supplied test labels are deliberately never inspected.
        record={'id':did,'text':text.strip(),'relevant':rel}
        if did in out:
            if out[did]!=record:raise ValueError(f'Conflicting repeated {split} ID {did}; inspect source data.')
            duplicates+=1;continue
        out[did]=record
    if not out:raise ValueError(f'Empty {split} data.')
    return out,{'exact_duplicate_rows_removed':duplicates,'blank_source_labels_ignored':blank_labels}
def normhash(text):return digest(' '.join(text.lower().split()))
def load_data(root):
    raw={};audit={}
    for split,(qfile,cfile) in FILES.items():
        queries,qa=records(read_json(Path(root)/qfile),split,True);candidates,ca=records(read_json(Path(root)/cfile),split,False)
        if len(candidates)<7:raise ValueError(f'{split}: candidate pool must contain at least 7 cases.')
        for q in queries.values():
            missing=set(q['relevant'] or [])-set(candidates)
            if missing:raise ValueError(f'{split}/{q["id"]}: relevance IDs outside its corresponding pool: {sorted(missing)[:5]}')
        raw[split]={'queries':queries,'candidates':candidates};audit[split]={'queries':qa,'candidates':ca}
    # Candidate overlaps are legitimate shared precedents. Do not delete candidate documents across pools.
    test=raw['test']['queries'];test_hashes={normhash(r['text']) for r in test.values()}
    removed_dev=[q for q,r in raw['dev']['queries'].items() if q in test or normhash(r['text']) in test_hashes]
    for q in removed_dev:del raw['dev']['queries'][q]
    held_ids=set(raw['dev']['queries'])|set(test);held_hashes={normhash(r['text']) for s in ('dev','test') for r in raw[s]['queries'].values()}
    removed_train=[q for q,r in raw['train']['queries'].items() if q in held_ids or normhash(r['text']) in held_hashes]
    for q in removed_train:del raw['train']['queries'][q]
    if any(not raw[s]['queries'] for s in FILES):raise ValueError('Query overlap filtering left an empty split.')
    audit.update(train_held_query_copies_removed=removed_train,dev_test_query_copies_removed=removed_dev,
                 candidate_id_overlap_counts={f'{a}_{b}':len(set(raw[a]['candidates'])&set(raw[b]['candidates'])) for a,b in [('train','dev'),('train','test'),('dev','test')]},
                 policy='Only train queries/relevance drive training. Dev is for selection. Test relevance is discarded. Candidate pools remain split-specific; shared precedents are allowed.')
    return raw,audit

STOP=set('a an the and or but if to of in on for by with is are was were be been being it its this that these those as at from has have had not no he she they their we our you your i court judgment case appellant respondent petitioner learned honble said shall would could also'.split())
WORD=re.compile(r'[a-z][a-z0-9_]*|\d+',re.I)
def terms(text):return [w for w in WORD.findall(text.lower()) if w not in STOP and len(w)>1]
class BM25:
    """Sparse inverted BM25. IDF is computed on each permitted candidate corpus, without relevance labels."""
    def __init__(self,max_features=200000,k1=1.2,b=.75):self.max_features=max_features;self.k1=k1;self.b=b
    def fit(self,texts):
        from sklearn.feature_extraction.text import CountVectorizer
        from scipy import sparse
        self.vectorizer=CountVectorizer(analyzer=terms,max_features=self.max_features,dtype=np.float32)
        X=self.vectorizer.fit_transform(texts).tocsr();N=X.shape[0];df=np.asarray((X>0).sum(0)).ravel()
        self.idf=np.log1p((N-df+.5)/(df+.5)).astype(np.float32);length=np.asarray(X.sum(1)).ravel();avg=max(1.,float(length.mean()))
        denominator=self.k1*(1-self.b+self.b*length/avg)
        X.data=(X.data*(self.k1+1)/(X.data+np.repeat(denominator,np.diff(X.indptr))))*self.idf[X.indices]
        self.matrix=X.tocsr();return self
    def scores(self,text):
        q=self.vectorizer.transform([text]).tocsr();q.data=np.log1p(q.data)
        # Bound very long query term sets by candidate IDF×query-TF; no labels are used.
        if q.nnz>512:
            keep=np.argsort(-(q.data*self.idf[q.indices]),kind='stable')[:512];q.data[np.setdiff1d(np.arange(q.nnz),keep)]=0;q.eliminate_zeros()
        return (q@self.matrix.T).toarray().ravel().astype(np.float32)
def top_indices(scores,n):return np.argsort(-scores,kind='stable')[:min(n,len(scores))]

def prepare_cache(root,tokenizer_path,output,settings):
    output=Path(output);output.mkdir(parents=True,exist_ok=True);manifest=output/'manifest.json'
    relevant_settings={k:settings[k] for k in ('chunk_tokens','chunk_overlap','max_chunks','lexical_topk','bm25_features')}
    identity=digest({'files':{name:file_hash(Path(root)/name) for pair in FILES.values() for name in pair},'tokenizer':file_hash(tokenizer_path),'settings':relevant_settings,'version':1})
    if manifest.exists():
        old=read_json(manifest)
        if old['fingerprint']!=identity:raise ValueError('Data/tokenization settings changed. Use a new PROJECT.')
        if (output/'tokens.i32').stat().st_size!=4*old['total_tokens']:raise ValueError('Incomplete token cache. Use a new PROJECT to rebuild safely.')
        if any(r['relevant'] is not None for r in old['splits']['test']['queries'].values()):raise ValueError('Unsafe cache contains test relevance.')
        return str(manifest)
    raw,audit=load_data(root);tok=Tokenizer.from_file(str(tokenizer_path));tok.no_padding();tok.no_truncation()
    pad=next((tok.token_to_id(x) for x in ('<pad>','[PAD]') if tok.token_to_id(x) is not None),None)
    bos=next((tok.token_to_id(x) for x in ('<bos>','[BOS]','<s>') if tok.token_to_id(x) is not None),None)
    if pad is None or bos is None or pad==bos:raise ValueError('Tokenizer must contain distinct existing PAD and BOS tokens.')
    unique={};docs={};splits={};total=0;df=np.zeros(tok.get_vocab_size(),dtype=np.int64);train_candidate_count=0
    tmp=output/'tokens.i32.tmp';span=settings['chunk_tokens'];stride=span-settings['chunk_overlap']
    if stride<=0:raise ValueError('Chunk overlap must be smaller than chunk size.')
    with tmp.open('wb') as f:
        for split,pool in raw.items():
            splits[split]={'queries':{},'candidates':{}}
            for kind in ('queries','candidates'):
                for did,r in sorted(pool[kind].items()):
                    sha=digest(r['text']);key=f'{split}/{kind}/{did}'
                    if sha not in unique:
                        ids=tok.encode(r['text'],add_special_tokens=False).ids or [bos];n=len(ids)
                        starts=list(range(0,max(1,n-span+1),stride));last=max(0,n-span)
                        if starts[-1]!=last:starts.append(last)
                        # Preserve all token IDs in the memmap; neural training uses a bounded selected subset.
                        np.asarray(ids,dtype='<i4').tofile(f);windows=[[total+a,min(span,n-a),a/max(1,n-1)] for a in starts]
                        unique[sha]={'offset':total,'tokens':n,'windows':windows};total+=n
                    meta=unique[sha]
                    docs[key]={**meta,'id':did,'text_sha256':sha}
                    splits[split][kind][did]={'key':key,**({'relevant':r['relevant']} if kind=='queries' else {})}
                    if split=='train' and kind=='candidates':
                        ids=tok.encode(r['text'],add_special_tokens=False).ids;df[np.unique(ids)]+=1;train_candidate_count+=1
            print('Tokenized split:',split,flush=True)
    tmp.replace(output/'tokens.i32');idf=np.log1p((train_candidate_count+.5)/(df+.5)).astype(np.float32)
    np.save(output/'token_idf.npy',idf);mm=np.memmap(output/'tokens.i32',mode='r',dtype='<i4')
    for d in docs.values():
        windows=d['windows'];scores=[float(idf[np.unique(mm[o:o+n])].mean()) for o,n,_ in windows]
        d['order']=list(dict.fromkeys([0,len(windows)-1]+sorted(range(len(windows)),key=lambda j:(-scores[j],j))))
    lexical={};baseline={};ceilings={}
    from evaluate_phase7_official import score
    for split,pool in raw.items():
        ids=sorted(pool['candidates']);lookup={c:i for i,c in enumerate(ids)};index=BM25(settings['bm25_features']).fit([pool['candidates'][i]['text'] for i in ids]);entries={};prediction={}
        for qid,r in sorted(pool['queries'].items()):
            scores=index.scores(r['text']);order=top_indices(scores,settings['lexical_topk']);entries[qid]={'ids':[ids[i] for i in order],'scores':[float(scores[i]) for i in order]};prediction[qid]=entries[qid]['ids'][:7]
            if split=='train':
                entries[qid]['extra_scores']={c:float(scores[lookup[c]]) for c in r['relevant']}
        lexical[split]=entries;atomic_pickle(output/f'bm25_{split}.pkl',index)
        if split!='test':
            gold={q:r['relevant'] for q,r in pool['queries'].items()};baseline[split]=score(gold,prediction)
            found=sum(len(set(gold[q])&set(entries[q]['ids'])) for q in gold);all_gold=sum(len(g) for g in gold.values());ceilings[split]={'lexical_shortlist_recall':found/max(1,all_gold),'micro_f1_at7_upper_bound_within_shortlist':2*sum(min(7,len(set(gold[q])&set(entries[q]['ids']))) for q in gold)/max(1,7*len(gold)+all_gold)}
        atomic_json(output/f'{split}_candidate_ids.json',ids)
        print('BM25 indexed:',split,'queries',len(entries),'candidates',len(ids),flush=True)
    atomic_json(output/'lexical.json',lexical);audit.update(counts={s:{k:len(v) for k,v in sp.items()} for s,sp in splits.items()},baseline_bm25=baseline,shortlist_ceilings=ceilings,
        total_unique_document_texts=len(unique),total_token_ids=total,neural_view=f'At most {settings["max_chunks"]} selected {span}-token windows per document. Full tokens are cached; not all are neurally encoded.')
    value={'fingerprint':identity,'tokenizer_path':str(tokenizer_path),'tokenizer_sha256':file_hash(tokenizer_path),'special':{'pad':pad,'bos':bos},'cache_dir':str(output),'docs':docs,'splits':splits,'total_tokens':total,'settings':relevant_settings,'audit':audit}
    atomic_json(manifest,value);atomic_json(output/'audit.json',audit);return str(manifest)

class Store:
    def __init__(self,manifest):
        self.manifest=read_json(manifest);self.root=Path(self.manifest['cache_dir']);self.docs=self.manifest['docs'];self.splits=self.manifest['splits']
        self.tokens=np.memmap(self.root/'tokens.i32',mode='r',dtype='<i4');self.idf=np.load(self.root/'token_idf.npy');self.lexical=read_json(self.root/'lexical.json')
        self.train_candidate_ids=sorted(self.splits['train']['candidates'])
    def chunks(self,key,count=None):
        d=self.docs[key];count=count or self.manifest['settings']['max_chunks'];order=sorted(d['order'][:count]);return [(self.tokens[d['windows'][i][0]:d['windows'][i][0]+d['windows'][i][1]].copy(),d['windows'][i][2]) for i in order]
    def pair(self,qkey,dkey,width):
        query,qset=self.query_view(qkey,width);best=None;bestscore=-1
        for offset,n,unique in self.window_terms(dkey):
            overlap=np.intersect1d(qset,unique,assume_unique=True);s=float(self.idf[overlap].sum())
            if s>bestscore:bestscore=s;best=self.tokens[offset:offset+n].copy()
        dcap=width-len(query)-3;doc=best[:dcap];bos=self.manifest['special']['bos']
        ids=np.concatenate([[bos],query,[bos],doc,[bos]]).astype(np.int64)
        segment=np.asarray([0]*(len(query)+2)+[1]*(len(doc)+1),dtype=np.int64)
        return ids,segment
    @functools.lru_cache(maxsize=2048)
    def query_view(self,key,width):
        flat=np.concatenate([a for a,_ in self.chunks(key)]);cap=(width-3)//2
        query=np.concatenate([flat[:cap//2],flat[-(cap-cap//2):]]) if len(flat)>cap else flat
        # Window selection can use all selected query chunks even though the joint neural pair is shorter.
        return query,np.unique(flat)
    @functools.lru_cache(maxsize=2048)
    def window_terms(self,key):
        return tuple((offset,n,np.unique(self.tokens[offset:offset+n])) for offset,n,_ in self.docs[key]['windows'])
'''
(CODE/'p7_data.py').write_text(module_source, encoding='utf-8')
print('Wrote p7_data.py')

In [ ]:
# p7_eval.py
module_source = r'''"""Shortlist reranking, exact official Micro-F1@7, and checked submission archives."""
import contextlib,json,math,time,zipfile
from pathlib import Path
import numpy as np
import torch
from evaluate_phase7_official import score,validate
from p7_data import atomic_json,digest,read_json
from p7_model import encode_batch,pairs_batch,similarity,merged_adapters

def metrics(store,split,pred):
    queries=store.splits[split]['queries'];gold={q:r['relevant'] for q,r in queries.items() if q in pred}
    if split=='test':raise ValueError('Test labels are never used for scoring.')
    errors=validate(pred,gold,set(store.splits[split]['candidates']))
    if errors:raise ValueError('; '.join(errors[:5]))
    raw=score(gold,pred)
    return {'micro_f1_at7':raw['primary_metric_1'],'micro_precision_at7':raw['secondary_metric_1'],'micro_recall_at7':raw['secondary_metric_2'],'tp':raw['tp'],'fp':raw['fp'],'fn':raw['fn'],'queries':len(gold)}
def normalized(x):
    x=np.asarray(x,dtype=np.float32);std=max(float(x.std()),1e-6);return (x-x.mean())/std
def predictions(raw,weight):
    return {q:ids[:7] for q,ids in ranking_lists(raw,weight).items()}
def ranking_lists(raw,weight):
    pred={}
    for q,r in raw.items():
        score=weight*normalized(r['neural'])+(1-weight)*normalized(r['lexical'])
        order=np.argsort(-score,kind='stable');pred[q]=[r['ids'][i] for i in order]
    return pred
def select_fusion(store,raw,weights=(.1,.25,.5,.75,1.)):
    candidates=[]
    for w in weights:
        pred=predictions(raw,w);m=metrics(store,'dev',pred);candidates.append({'neural_weight':float(w),'metrics':m})
    # Prefer the stronger neural contribution on an exact tie; the sparse-only baseline is reported separately.
    best=max(candidates,key=lambda x:(x['metrics']['micro_f1_at7'],x['neural_weight']))
    return {**best,'candidates':candidates,'scope':'Validation selection estimate, not an independent test score.'}
def query_ids(store,split,limit=None,seed=781):
    ids=sorted(store.splits[split]['queries'],key=lambda q:digest([seed,q]))
    return ids[:limit] if limit else ids
def shortlist(store,split,qid,cfg,base_rankings=None):
    lex=store.lexical[split][qid];ids=lex['ids'];values=dict(zip(ids,lex['scores']))
    if cfg['kind']!='cross':return ids,lex['scores']
    k=min(cfg['rerank_topk'],len(ids))
    # Cross-encoder reranks a union of lexical and trained first-stage rankings, all inside this split's lexical pool.
    dense=base_rankings.get(qid,ids) if base_rankings else ids
    selected=list(dict.fromkeys(ids[:max(7,k//2)]+[i for i in dense[:k//2] if i in values]+ids))[:k]
    return selected,[values[i] for i in selected]
@torch.inference_mode()
def teacher_scores(model,store,qids,cfg,device,count=16):
    # Gold injection is restricted to TRAIN teacher supervision, never validation/test retrieval.
    model.eval();out={};amp=device.type=='cuda';selection_path=Path(cfg['run'])/'full_dev_selection.json'
    weight=read_json(selection_path)['neural_weight'] if selection_path.is_file() else 1.
    with merged_adapters(model):
        for qid in qids:
            row=store.splits['train']['queries'][qid];lex=store.lexical['train'][qid];gold=row['relevant']
            if not gold:continue
            ids=list(dict.fromkeys(gold+lex['ids']))[:max(count,len(gold)+1)]
            if not set(ids)-set(gold):continue
            scores=dict(zip(lex['ids'],lex['scores']));scores.update(lex.get('extra_scores',{}));qkey=row['key'];dkeys=[store.splits['train']['candidates'][c]['key'] for c in ids]
            if cfg['kind']=='cross':
                batch=pairs_batch(store,[qkey]*len(ids),dkeys,[scores.get(c,0.) for c in ids],cfg,device)
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):values=model.rank_pairs(batch)
            else:
                batch=encode_batch(store,[qkey]+dkeys,[0]+[1]*len(ids),cfg,device)
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):g,ch,m=model.encode(batch)
                values=similarity((g[:1],ch[:1],m[:1]),(g[1:],ch[1:],m[1:]),cfg['kind']).squeeze(0)/cfg['temperature']
            neural=values.cpu().float().numpy();lexical=np.asarray([scores.get(c,0.) for c in ids],dtype=np.float32)
            fused=4*(weight*normalized(neural)+(1-weight)*normalized(lexical))
            out[qid]={'ids':ids,'scores':fused.tolist()}
    return out
@torch.inference_mode()
def rank(model,store,split,qids,cfg,device,base_rankings=None):
    model.eval();raw={};started=time.perf_counter();amp=device.type=='cuda';pools={q:shortlist(store,split,q,cfg,base_rankings) for q in qids}
    with merged_adapters(model):
        if cfg['kind']=='cross':
            for qi,qid in enumerate(qids):
                ids,lex=pools[qid];scores=[];qkey=store.splits[split]['queries'][qid]['key']
                for a in range(0,len(ids),cfg['inference_pairs']):
                    local=ids[a:a+cfg['inference_pairs']];dkeys=[store.splits[split]['candidates'][c]['key'] for c in local]
                    batch=pairs_batch(store,[qkey]*len(local),dkeys,lex[a:a+len(local)],cfg,device)
                    with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):s=model.rank_pairs(batch)
                    scores.extend(s.cpu().float().tolist())
                raw[qid]={'ids':ids,'lexical':lex,'neural':scores}
                if (qi+1)%25==0:print(json.dumps({'reranked_queries':qi+1,'total':len(qids),'seconds':time.perf_counter()-started}),flush=True)
        else:
            # Encode each candidate only once per rank/validation pass, not once per query–candidate pair.
            cids=sorted({c for q in qids for c in pools[q][0]});vectors=[];chunks=[];masks=[];batch_size=cfg['index_batch']
            for a in range(0,len(cids),batch_size):
                local=cids[a:a+batch_size];keys=[store.splits[split]['candidates'][c]['key'] for c in local];batch=encode_batch(store,keys,[1]*len(local),cfg,device)
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):g,ch,ma=model.encode(batch)
                vectors.append(g.cpu().half());chunks.append(ch.cpu().half());masks.append(ma.cpu())
            if not cids:return raw
            g=torch.cat(vectors);ch=torch.cat(chunks);ma=torch.cat(masks);lookup={c:i for i,c in enumerate(cids)}
            for qi,qid in enumerate(qids):
                ids,lex=pools[qid];key=store.splits[split]['queries'][qid]['key'];batch=encode_batch(store,[key],[0],cfg,device)
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):q=model.encode(batch)
                idx=torch.tensor([lookup[c] for c in ids]);d=(g[idx].to(device).float(),ch[idx].to(device).float(),ma[idx].to(device))
                with torch.autocast(device_type=device.type,enabled=False):scores=similarity(q,d,cfg['kind']).squeeze(0).cpu().float().tolist()
                raw[qid]={'ids':ids,'lexical':lex,'neural':scores}
                if (qi+1)%100==0:print(json.dumps({'ranked_queries':qi+1,'total':len(qids),'seconds':time.perf_counter()-started}),flush=True)
    if any(not np.isfinite(r['neural']).all() for r in raw.values()):raise FloatingPointError('Nonfinite retrieval scores.')
    return raw
def pack(project,name,pred,store):
    queries=store.splits['test']['queries'];gold={q:[] for q in queries};pool=set(store.splits['test']['candidates']);errors=validate(pred,gold,pool)
    if errors:raise ValueError('; '.join(errors[:10]))
    folder=Path(project)/'submissions'/name;atomic_json(folder/'prediction.json',pred);archive=folder.with_suffix('.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:z.write(folder/'prediction.json','prediction.json')
    with zipfile.ZipFile(archive) as z:
        if z.namelist()!=['prediction.json']:raise ValueError('ZIP must contain prediction.json at its root.')
    return {'name':name,'submission_zip':str(archive),'prediction_sha256':digest(pred),'queries':len(pred)}
'''
(CODE/'p7_eval.py').write_text(module_source, encoding='utf-8')
print('Wrote p7_eval.py')

In [ ]:
# p7_model.py
module_source = r'''"""Own Phase-1 retrieval backbones, multi-positive contrastive loss and pair ranking."""
import contextlib,math,types
from dataclasses import fields
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from legal_models import LegalLM,ModelConfig,Attention

class LoRMA(nn.Module):
    def __init__(self,base,rank=16,alpha=32):
        super().__init__();self.base=base;base.requires_grad_(False);self.scale=alpha/rank
        self.A=nn.Linear(base.out_features,rank,bias=False);self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=.02);nn.init.zeros_(self.B.weight)
    def forward(self,x):
        y=self.base(x);return y+self.scale*self.B(self.A(y))
class IA3(nn.Module):
    def __init__(self,base,on_input=False):
        super().__init__();self.base=base;base.requires_grad_(False);self.on_input=on_input
        self.gain=nn.Parameter(torch.ones(base.in_features if on_input else base.out_features))
    def forward(self,x):return self.base(x*self.gain) if self.on_input else self.base(x)*self.gain
@contextlib.contextmanager
def merged_adapters(model):
    replaced=[]
    with torch.no_grad():
        for name,m in list(model.named_modules()):
            if not isinstance(m,(LoRMA,IA3)):continue
            parent,attr=name.rsplit('.',1);parent=model.get_submodule(parent);base=m.base
            layer=nn.Linear(base.in_features,base.out_features,bias=base.bias is not None,device=base.weight.device,dtype=base.weight.dtype)
            if isinstance(m,LoRMA):weight=base.weight+m.scale*m.B.weight@(m.A.weight@base.weight)
            else:weight=base.weight*(m.gain[None,:] if m.on_input else m.gain[:,None])
            layer.weight.copy_(weight)
            if base.bias is not None:layer.bias.copy_(base.bias if isinstance(m,IA3) and m.on_input else (base.bias*m.gain if isinstance(m,IA3) else base.bias+m.scale*m.B.weight@(m.A.weight@base.bias)))
            layer.requires_grad_(False);setattr(parent,attr,layer);replaced.append((parent,attr,m))
    try:yield
    finally:
        for parent,attr,m in replaced:setattr(parent,attr,m)
def masked_attention(self,x,memory=None,past=None,position=0,use_cache=False):
    if past is not None or use_cache:raise ValueError('Retrieval encodes complete independent chunks; generation caches are not used.')
    b,t,_=x.shape;source=x if memory is None else memory;s=source.size(1)
    q=self.q(x).view(b,t,self.heads,self.hd).transpose(1,2);k=self.k(source).view(b,s,self.kv_heads,self.hd).transpose(1,2);v=self.v(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    if self.rotary is not None:q=self.rotary(q);k=self.rotary(k)
    if self.kv_heads!=self.heads:k=k.repeat_interleave(self.heads//self.kv_heads,1);v=v.repeat_interleave(self.heads//self.kv_heads,1)
    mask=self._valid[:,None,None,:s]
    if self.causal:
        mask=mask&(torch.arange(s,device=x.device)[None,:]<=torch.arange(t,device=x.device)[:,None])[None,None]
    # Encoder-style bidirectional context for retrieval. GQA projections and all Phase-1 tensors are preserved.
    out=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=self.dropout if self.training else 0.)
    return self.out(out.transpose(1,2).contiguous().view(b,t,-1)),None
def load_parent(path,tokenizer_sha,allow_tiny=False):
    saved=torch.load(path,map_location='cpu',weights_only=False)
    if saved.get('training_origin')!='random_initialization_official_phase1_only':raise ValueError('An original own Phase-1 checkpoint is required.')
    if saved.get('tokenizer_sha256')!=tokenizer_sha:raise ValueError('Checkpoint/tokenizer SHA256 mismatch.')
    c=ModelConfig(**{k:v for k,v in saved['model_config'].items() if k in {f.name for f in fields(ModelConfig)}});c.gradient_checkpointing=False
    model=LegalLM(c);model.load_state_dict(saved['model'],strict=True);n=sum(p.numel() for p in model.parameters())
    if not allow_tiny and not 50_000_000<=n<=300_000_000:raise ValueError(f'Original foundation has {n:,} parameters, outside 50M–300M.')
    return model
class Retriever(nn.Module):
    def __init__(self,backbone,cfg):
        super().__init__();self.backbone=backbone;self.cfg=cfg;self.cross=cfg['kind']=='cross';self.native=backbone.config.architecture=='encoder_decoder';d=backbone.config.dim
        method=cfg['method'];backbone.requires_grad_(method in ('full','kd'))
        if method=='kd':
            indices=np.linspace(0,len(backbone.blocks)-1,max(1,len(backbone.blocks)//2)).round().astype(int).tolist()
            backbone.blocks=nn.ModuleList([backbone.blocks[i] for i in indices]);backbone.config.layers=len(indices);self.student_layers=indices
        elif method=='partial':
            for block in list(backbone.blocks)[-cfg.get('unfrozen_layers',2):]:block.requires_grad_(True)
            backbone.final_norm.requires_grad_(True)
        elif method in ('lorma','ia3'):
            for name,module in list(backbone.named_modules()):
                if not isinstance(module,nn.Linear):continue
                leaf=name.rsplit('.',1)[-1]
                use=leaf in ({'q','k','v','out','gate','up','down'} if method=='lorma' else {'k','v','down'})
                if use:
                    parent,attr=name.rsplit('.',1);replacement=LoRMA(module,cfg.get('rank',16),cfg.get('alpha',32)) if method=='lorma' else IA3(module,leaf=='down')
                    setattr(backbone.get_submodule(parent),attr,replacement)
        elif method=='prompt':self.prompt=nn.Parameter(backbone.embedding.weight.detach()[:cfg.get('prompt_tokens',32)].clone())
        elif method!='full':
            if method!='kd':raise ValueError(f'Unknown method {method}')
        for m in backbone.modules():
            if isinstance(m,Attention):m.forward=types.MethodType(masked_attention,m);m.causal=False;m.window=0;m.dropout=cfg.get('dropout',.05)
            if isinstance(m,nn.Dropout):m.p=cfg.get('dropout',.05)
        self.checkpointing=cfg.get('checkpointing',True)
        if self.native:
            self.readouts=nn.Parameter(backbone.embedding.weight.detach()[cfg['bos']][None].repeat(4,1)+.01*torch.randn(4,d))
        if self.cross:
            self.segment=nn.Embedding(2,d);nn.init.zeros_(self.segment.weight)
            self.rank_head=nn.Sequential(nn.LayerNorm(4*d+3),nn.Linear(4*d+3,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1))
        else:
            h=cfg.get('embedding_dim',256);self.token_gate=nn.Linear(d,1,bias=False)
            self.project=nn.Sequential(nn.LayerNorm(2*d),nn.Linear(2*d,h),nn.GELU(),nn.Linear(h,h))
            self.roles=nn.Parameter(torch.zeros(2,h));self.chunk_gate=nn.Linear(h+2,1)
    def _block(self,block,x,valid,memory=None,memory_valid=None):
        if memory is None:block.attn._valid=valid;return block(x)
        block.self_attn._valid=valid;block.cross_attn._valid=memory_valid;return block(x,memory)
    def hidden(self,ids,valid,segments=None):
        x=self.backbone.embedding(ids)
        if segments is not None:x=x+self.segment(segments)
        if hasattr(self,'prompt'):
            x=torch.cat([self.prompt[None].expand(len(ids),-1,-1),x],1)
            valid=torch.cat([torch.ones(len(ids),len(self.prompt),device=ids.device,dtype=torch.bool),valid],1)
        if hasattr(self.backbone,'rnn'):
            # Pack right padding; keep cuDNN LSTM outside activation-checkpoint recomputation.
            packed=nn.utils.rnn.pack_padded_sequence(x,valid.sum(1).cpu(),batch_first=True,enforce_sorted=False)
            y,_=self.backbone.rnn(packed);y,_=nn.utils.rnn.pad_packed_sequence(y,batch_first=True,total_length=x.shape[1]);x=x+self.backbone.rnn_norm(y)
        memory=None;memory_valid=None
        if self.native:
            for block in self.backbone.encoder:
                def f(x,valid,block=block):return self._block(block,x,valid)
                x=checkpoint(f,x,valid,use_reentrant=False) if self.training and self.checkpointing else f(x,valid)
            memory=self.backbone.encoder_norm(x);memory_valid=valid;x=self.readouts[None].expand(len(ids),-1,-1);valid=torch.ones(x.shape[:2],device=x.device,dtype=torch.bool)
        for block in self.backbone.blocks:
            def f(x,valid,block=block,memory=memory,memory_valid=memory_valid):return self._block(block,x,valid,memory,memory_valid)
            x=checkpoint(f,x,valid,use_reentrant=False) if self.training and self.checkpointing else f(x,valid)
        return self.backbone.final_norm(x),valid
    def encode(self,batch):
        ids=batch['ids'];valid=batch['valid'];pooled=[];step=self.cfg.get('chunk_microbatch',4)
        for a in range(0,len(ids),step):
            h,m=self.hidden(ids[a:a+step],valid[a:a+step]);gate=self.token_gate(h)
            with torch.autocast(device_type=h.device.type,enabled=False):
                weights=torch.softmax(gate.squeeze(-1).float().masked_fill(~m,float('-inf')),dim=-1)
                attention=(h.float()*weights[:,:,None]).sum(1);mean=(h.float()*m[:,:,None]).sum(1)/m.sum(1)[:,None]
            pooled.append(self.project(torch.cat([attention,mean],-1)))
        vectors=torch.cat(pooled);B=batch['documents'];M=batch['max_chunks'];owner=batch['owner'];slot=batch['slot'];index=owner*M+slot
        padded=vectors.new_zeros(B*M,vectors.shape[-1]).index_copy(0,index,vectors).reshape(B,M,-1)
        mask=torch.zeros(B*M,device=ids.device,dtype=torch.bool).index_fill(0,index,True).reshape(B,M)
        padded=F.normalize(padded.float()+self.roles[batch['roles']][:,None,:],dim=-1)*mask[:,:,None]
        position=batch['position'];gate=self.chunk_gate(torch.cat([padded,position],-1))
        with torch.autocast(device_type=ids.device.type,enabled=False):
            weight=torch.softmax(gate.squeeze(-1).float().masked_fill(~mask,float('-inf')),dim=-1)
            global_vector=F.normalize((padded.float()*weight[:,:,None]).sum(1),dim=-1)
        return global_vector,padded,mask
    def rank_pairs(self,batch):
        scores=[];step=self.cfg.get('pair_microbatch',4)
        for a in range(0,len(batch['ids']),step):
            ids=batch['ids'][a:a+step];valid=batch['valid'][a:a+step];segment=batch['segments'][a:a+step]
            h,_=self.hidden(ids,valid,segment);q=valid&(segment==0);d=valid&(segment==1)
            with torch.autocast(device_type=h.device.type,enabled=False):
                qm=(h.float()*q[:,:,None]).sum(1)/q.sum(1).clamp_min(1)[:,None];dm=(h.float()*d[:,:,None]).sum(1)/d.sum(1).clamp_min(1)[:,None]
                features=torch.cat([qm,dm,qm*dm,(qm-dm).abs(),batch['lexical'][a:a+step]],-1)
            scores.append(self.rank_head(features).squeeze(-1))
        return torch.cat(scores).float()
    def forward(self,batch):
        if batch['mode']=='encode':return self.encode(batch)
        if batch['mode']=='pairs':return self.rank_pairs(batch)
        if self.cross:
            logits=self.rank_pairs(batch).reshape(batch['query_count'],-1);gold=batch['positive'].bool()
            with torch.autocast(device_type=logits.device.type,enabled=False):
                rank=multi_positive_loss(logits.float(),gold)
                binary=F.binary_cross_entropy_with_logits(logits.float(),gold.float())
            return .8*rank+.2*binary
        global_v,chunks,mask=self.encode(batch);nq=batch['query_count'];q=(global_v[:nq],chunks[:nq],mask[:nq]);d=(global_v[nq:],chunks[nq:],mask[nq:])
        # Cross-rank in-batch negatives retain gradient paths through gathered candidate embeddings.
        if torch.distributed.is_initialized():
            from torch.distributed.nn.functional import all_gather
            # One differentiable collective keeps global/chunk backward ordering identical across ranks.
            packed=torch.cat([d[0][:,None,:],d[1]],dim=1);packed=torch.cat(all_gather(packed),0)
            d=(packed[:,0],packed[:,1:],torch.cat(all_gather(d[2].float()),0).bool())
        with torch.autocast(device_type=global_v.device.type,enabled=False):
            scores=similarity(q,d,self.cfg['kind'])/self.cfg.get('temperature',.05)
            loss=multi_positive_loss(scores,batch['positive'])
            if 'teacher_scores' in batch:
                indices=batch['teacher_columns'];student=scores.gather(1,indices);teacher=batch['teacher_scores'].float();T=self.cfg.get('kd_temperature',2.)
                kd=F.kl_div(F.log_softmax(student/T,-1),F.softmax(teacher/T,-1),reduction='batchmean')*T*T
                loss=(1-self.cfg.get('kd_weight',.5))*loss+self.cfg.get('kd_weight',.5)*kd
        return loss
    def report(self):return {'architecture':self.backbone.config.architecture,'kind':self.cfg['kind'],'method':self.cfg['method'],'total_parameters':sum(p.numel() for p in self.parameters()),'trainable_parameters':sum(p.numel() for p in self.parameters() if p.requires_grad)}
def multi_positive_loss(scores,positive):
    if not positive.any(1).all():raise ValueError('Each training query needs at least one positive column.')
    # Sum probability mass over all known relevant IDs, including other queries' positives.
    return (torch.logsumexp(scores,-1)-torch.logsumexp(scores.masked_fill(~positive,float('-inf')),-1)).mean()
def similarity(q,d,kind='dense'):
    if kind!='late':return q[0].float()@d[0].float().T
    # Chunk-level MaxSim, not token-level ColBERT or its compression/index implementation.
    interaction=torch.einsum('qih,djh->qdij',q[1].float(),d[1].float())
    maximum=interaction.masked_fill(~d[2][None,:,None,:],float('-inf')).max(-1).values
    late=(maximum*q[2][:,None,:]).sum(-1)/q[2].sum(-1)[:,None]
    # Include the learned global path so the complete document pooling head is trained too.
    return .8*late+.2*(q[0].float()@d[0].float().T)

def encode_batch(store,keys,roles,cfg,device):
    chunks=[];owners=[];slots=[];positions=[]
    M=cfg['max_chunks'];position=np.zeros((len(keys),M,2),dtype=np.float32)
    for i,key in enumerate(keys):
        for j,(ids,p) in enumerate(store.chunks(key,M)):
            chunks.append(ids);owners.append(i);slots.append(j);position[i,j]=[p,min(1.,len(ids)/cfg['chunk_tokens'])]
    T=max(map(len,chunks));pad=store.manifest['special']['pad'];ids=torch.full((len(chunks),T),pad,device=device,dtype=torch.long);valid=torch.zeros_like(ids,dtype=torch.bool)
    for i,tokens in enumerate(chunks):ids[i,:len(tokens)]=torch.as_tensor(tokens.astype(np.int64),device=device);valid[i,:len(tokens)]=True
    return {'mode':'encode','ids':ids,'valid':valid,'owner':torch.tensor(owners,device=device),'slot':torch.tensor(slots,device=device),
            'roles':torch.tensor(roles,device=device),'position':torch.tensor(position,device=device),'documents':len(keys),'max_chunks':M}
def pairs_batch(store,qkeys,dkeys,lexical,cfg,device):
    pairs=[store.pair(q,d,cfg['pair_tokens']) for q,d in zip(qkeys,dkeys)];T=max(len(ids) for ids,_ in pairs)
    ids=torch.full((len(pairs),T),store.manifest['special']['pad'],device=device,dtype=torch.long);valid=torch.zeros_like(ids,dtype=torch.bool);segments=torch.zeros_like(ids)
    features=[]
    for i,((tokens,segment),score) in enumerate(zip(pairs,lexical)):
        ids[i,:len(tokens)]=torch.tensor(tokens,device=device);valid[i,:len(tokens)]=True;segments[i,:len(tokens)]=torch.tensor(segment,device=device)
        q=tokens[segment==0];d=tokens[segment==1];overlap=len(set(q)&set(d))/max(1,len(set(q)))
        features.append([math.log1p(max(0.,score))/5,overlap,min(len(tokens)/cfg['pair_tokens'],1.)])
    return {'mode':'pairs','ids':ids,'valid':valid,'segments':segments,'lexical':torch.tensor(features,device=device,dtype=torch.float32)}
'''
(CODE/'p7_model.py').write_text(module_source, encoding='utf-8')
print('Wrote p7_model.py')

In [ ]:
# p7_run.py
module_source = r'''"""Two-GPU DDP training and split-specific retrieval workers."""
import argparse,contextlib,json,math,os,random,time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DistributedSampler,DataLoader
from p7_data import Store,read_json,atomic_json,file_hash,digest
from p7_model import Retriever,load_parent,encode_batch,pairs_batch
from p7_eval import rank,select_fusion,query_ids,teacher_scores

def save(path,x):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp');torch.save(x,tmp);tmp.replace(path)
def initialize():
    rank=int(os.environ.get('RANK',0));world=int(os.environ.get('WORLD_SIZE',1));local=int(os.environ.get('LOCAL_RANK',0));device=torch.device(f'cuda:{local}' if torch.cuda.is_available() else 'cpu')
    if device.type=='cuda':torch.cuda.set_device(device)
    if world>1:dist.init_process_group('nccl' if device.type=='cuda' else 'gloo',**({'device_id':device} if device.type=='cuda' else {}))
    return rank,world,device
def barrier(device):
    if dist.is_initialized():dist.barrier(device_ids=[device.index] if device.type=='cuda' else None)
def all_finite(value,device):
    flag=torch.isfinite(value.detach()).all().to(device=device,dtype=torch.int32)
    if dist.is_initialized():dist.all_reduce(flag,op=dist.ReduceOp.MIN)
    return bool(flag.item())
def finish_update(model,optimizer,scaler,params,device,stage):
    """Unscale once at the effective-batch boundary; all ranks step or skip together."""
    scaler.unscale_(optimizer)
    gradients=[p.grad for p in params if p.grad is not None]
    if not gradients:raise RuntimeError('No trainable gradients were produced.')
    norm=torch.nn.utils.get_total_norm(gradients,error_if_nonfinite=False)
    if not bool(torch.isfinite(norm)) and all(bool(torch.isfinite(g).all()) for g in gradients):
        # A float32 norm reduction itself can overflow even when every gradient element is finite.
        norm=torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(g.detach().double()) for g in gradients]))
    if not all_finite(norm,device):
        bad=[n for n,p in model.named_parameters() if p.grad is not None and not bool(torch.isfinite(p.grad).all())][:8]
        previous=float(scaler.get_scale());new_scale=previous*.5
        print(json.dumps({'event':'amp_gradient_overflow','stage':stage,'rank':dist.get_rank() if dist.is_initialized() else 0,
                          'scale_before':previous,'scale_after':new_scale,'nonfinite_gradient_parameters':bad,
                          'optimizer_step':'skipped on all ranks; parameters unchanged'}),flush=True)
        if not scaler.is_enabled():raise FloatingPointError(f'{stage}: nonfinite unscaled gradients with AMP disabled: {bad}')
        # Reset per-optimizer scaler state without updating weights, even if only a peer rank overflowed.
        scaler.update(new_scale=new_scale);optimizer.zero_grad(set_to_none=True)
        return float('inf'),False
    torch.nn.utils.clip_grads_with_norm_(params,1.,norm)
    scaler.step(optimizer);scaler.update()
    return float(norm),True
def preflight_update(wrapped,model,optimizer,scaler,params,batch,device,amp,stage,max_attempts=12):
    for attempt in range(max_attempts):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):loss=wrapped(batch)
        if not all_finite(loss,device):
            print(json.dumps({'event':'nonfinite_forward_loss','stage':stage,'rank':dist.get_rank() if dist.is_initialized() else 0,
                              'local_loss':str(float(loss.detach())),'amp_scale':float(scaler.get_scale())}),flush=True)
            raise FloatingPointError(f'{stage}: forward loss is nonfinite; this is not a loss-scaler-only overflow.')
        scaler.scale(loss).backward()
        # Watch a matrix with a nonzero gradient, not an arbitrary softmax bias whose gradient can be exactly zero.
        watch=next((p for p in params if p.grad is not None and p.ndim>1 and bool(torch.isfinite(p.grad).all()) and bool(p.grad.ne(0).any())),None)
        before=watch.detach().clone() if watch is not None else None
        norm,stepped=finish_update(model,optimizer,scaler,params,device,stage)
        if stepped:
            if watch is None or torch.equal(before,watch.detach()):raise ValueError('Preflight optimizer produced no detectable trainable matrix update.')
            return {'loss':float(loss.detach()),'gradient_norm':norm,'overflow_retries':attempt,'amp_scale':float(scaler.get_scale())}
    raise FloatingPointError(f'{stage}: gradients stayed nonfinite after {max_attempts} attempts. Last AMP scale={scaler.get_scale()}; inspect the printed parameter diagnostics.')
def build(cfg,store,device):
    parent=load_parent(cfg['parent'],store.manifest['tokenizer_sha256'],cfg.get('allow_tiny',False))
    if parent.config.architecture!=cfg['architecture']:raise ValueError('Parent architecture does not match experiment.')
    if max(cfg['chunk_tokens']+cfg.get('prompt_tokens',0),cfg['pair_tokens'])>parent.config.max_positions:raise ValueError('Retrieval sequence exceeds original position limit.')
    return Retriever(parent,cfg).to(device)
def base_rankings(cfg,split):
    if not cfg.get('base_run'):return None
    path=Path(cfg['base_run'])/f'{split}_rankings.json'
    if not path.is_file():raise FileNotFoundError(f'Complete run 01 before cross-encoder ranking: {path}')
    return read_json(path)
def training_queries(store):
    pool=store.splits['train']['candidates'];return [q for q,r in store.splits['train']['queries'].items() if r['relevant'] and len(set(r['relevant']))<len(pool)]
def samples(store,qids,cfg,epoch,hard=None,kd=None):
    result=[];pool=store.train_candidate_ids;n=cfg['negatives']
    for qid in qids:
        row=store.splits['train']['queries'][qid];gold=set(row['relevant']);rng=random.Random(int(digest([cfg['seed'],epoch,qid])[:16],16));teacher=kd.get(qid) if kd else None
        positives=sorted(gold & set(teacher['ids'])) if teacher else sorted(gold)
        if not positives:raise ValueError(f'{qid}: no teacher-supervised positive.')
        positive=positives[epoch%len(positives)];neg_pool=([c for c in teacher['ids'] if c not in gold] if teacher else [c for c in (hard or {}).get(qid,store.lexical['train'][qid]['ids']) if c not in gold])
        if not neg_pool:neg_pool=[c for c in pool if c not in gold]
        negatives=rng.sample(neg_pool,min(n,len(neg_pool)))
        # Mix one random negative to reduce overfitting to a single static hard-negative list.
        if not teacher and len(negatives)==n and n>1:
            blocked=gold|set(negatives[:-1]);draw=None
            for _ in range(32):
                candidate=rng.choice(pool)
                if candidate not in blocked:draw=candidate;break
            if draw is not None:negatives[-1]=draw
        while len(negatives)<n:negatives.append(rng.choice(neg_pool))
        ids=[positive]+negatives;lex=store.lexical['train'][qid];lexscores=dict(zip(lex['ids'],lex['scores']));lexscores.update(lex.get('extra_scores',{}))
        result.append({'qid':qid,'cids':ids,'gold':gold,'lexical':[lexscores.get(c,0.) for c in ids],
                       'teacher_scores':[dict(zip(teacher['ids'],teacher['scores']))[c] for c in ids] if teacher else None})
    return result
def training_batch(store,qids,cfg,epoch,device,hard=None,kd=None):
    rows=samples(store,qids,cfg,epoch,hard,kd);qkeys=[store.splits['train']['queries'][r['qid']]['key'] for r in rows]
    cids=[c for r in rows for c in r['cids']];dkeys=[store.splits['train']['candidates'][c]['key'] for c in cids]
    if cfg['kind']=='cross':
        batch=pairs_batch(store,[q for q,r in zip(qkeys,rows) for _ in r['cids']],dkeys,[x for r in rows for x in r['lexical']],cfg,device)
        batch.update(mode='loss',query_count=len(rows),positive=torch.tensor([[c in r['gold'] for c in r['cids']] for r in rows],device=device))
        return batch
    batch=encode_batch(store,qkeys+dkeys,[0]*len(qkeys)+[1]*len(dkeys),cfg,device);global_ids=cids;rank_id=0
    if dist.is_initialized():
        rank_id=dist.get_rank();gathered=[None]*dist.get_world_size();dist.all_gather_object(gathered,cids);global_ids=sum(gathered,[])
    batch.update(mode='loss',query_count=len(rows),positive=torch.tensor([[c in r['gold'] for c in global_ids] for r in rows],device=device))
    if kd:
        batch['teacher_scores']=torch.tensor([r['teacher_scores'] for r in rows],device=device,dtype=torch.float32)
        width=cfg['negatives']+1;batch['teacher_columns']=torch.arange(len(cids),device=device).reshape(len(rows),width)+rank_id*len(cids)
    return batch
def main():
    ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);ap.add_argument('--action',choices=['train','preflight','rank','teach'],default='train');ap.add_argument('--split',default='test');ap.add_argument('--output');args=ap.parse_args()
    cfg=read_json(args.config);store=Store(cfg['manifest']);rank_id,world,device=initialize();amp=device.type=='cuda'
    random.seed(cfg['seed']+rank_id);np.random.seed(cfg['seed']+rank_id);torch.manual_seed(cfg['seed'])
    if amp:torch.backends.cudnn.benchmark=True
    run=Path(cfg['run']);run.mkdir(parents=True,exist_ok=True)
    lineage={'config':cfg,'world_size':world,'parent_sha256':file_hash(cfg['parent']),'data_fingerprint':store.manifest['fingerprint'],
             'code_sha256':{p.name:file_hash(p) for p in Path(__file__).parent.glob('*.py')}}
    if cfg.get('hard_file'):lineage['hard_cache_sha256']=file_hash(cfg['hard_file']) if Path(cfg['hard_file']).is_file() else None
    if cfg.get('base_run'):lineage['base_rankings_sha256']={s:file_hash(Path(cfg['base_run'])/f'{s}_rankings.json') for s in ('train','dev','test') if (Path(cfg['base_run'])/f'{s}_rankings.json').is_file()}
    if cfg['method']=='kd':lineage['kd_cache_sha256']=file_hash(cfg['kd_file']) if Path(cfg['kd_file']).is_file() else None
    signature=digest(lineage);model=build(cfg,store,device);report=model.report();bestpath=run/'best.pt'
    if args.action in ('rank','teach'):
        saved=torch.load(bestpath,map_location='cpu',weights_only=False)
        if saved['signature']!=signature:raise ValueError('Checkpoint/input/code provenance mismatch.')
        model.load_state_dict(saved['model'],strict=True);del saved
        ids=query_ids(store,'train' if args.action=='teach' else args.split,seed=cfg['seed'])
        if args.action=='teach':raw=teacher_scores(model,store,ids[rank_id::world],cfg,device,cfg['teacher_candidates'])
        else:raw=rank(model,store,args.split,ids[rank_id::world],cfg,device,base_rankings(cfg,args.split))
        output=Path(args.output);atomic_json(str(output)+f'.rank{rank_id}.json',raw);barrier(device)
        if rank_id==0:
            merged={}
            for r in range(world):
                part=read_json(str(output)+f'.rank{r}.json')
                if set(part)&set(merged):raise ValueError('Duplicate distributed query output.')
                merged.update(part)
            if args.action=='rank' and set(merged)!=set(ids):raise ValueError('Missing query rankings.')
            atomic_json(output,merged)
        barrier(device)
        if dist.is_initialized():dist.destroy_process_group()
        return
    train=training_queries(store)
    if not train:raise ValueError('No training query has both a positive and a possible negative.')
    hard=None;kd=None
    if cfg.get('hard_file') and Path(cfg['hard_file']).is_file():
        h=read_json(cfg['hard_file'])
        if h['data_fingerprint']!=store.manifest['fingerprint']:raise ValueError('Hard negatives belong to different data.')
        hard=h['rankings']
    if cfg['method']=='kd' and args.action=='train':
        k=read_json(cfg['kd_file'])
        if k['data_fingerprint']!=store.manifest['fingerprint'] or not set(k['scores'])<=set(train):raise ValueError('Unsafe teacher cache; only eligible TRAIN queries may be distilled.')
        if k['teacher_checkpoint_sha256']!=file_hash(k['teacher_checkpoint']):raise ValueError('Teacher checkpoint changed.')
        kd=k['scores'];train=[q for q in train if q in kd]
        if not train:raise ValueError('KD cache contains no trainable queries.')
    wrapped=DDP(model,device_ids=[device.index] if amp else None,broadcast_buffers=False) if world>1 else model
    params=[p for p in model.parameters() if p.requires_grad];optimizer=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=.01,**({'fused':True} if amp else {}))
    initial_scale=float(cfg.get('amp_init_scale',1024.))
    if args.action=='train' and (run/'preflight.json').is_file():
        initial_scale=min(initial_scale,float(read_json(run/'preflight.json').get('amp_scale',initial_scale)))
    scaler=torch.amp.GradScaler(device.type,init_scale=initial_scale,enabled=amp)
    if args.action=='preflight':
        qids=[train[(rank_id*cfg['micro_batch']+i)%len(train)] for i in range(cfg['micro_batch'])];model.train();times=[];retry_count=0
        for i in range(3):
            optimizer.zero_grad(set_to_none=True);batch=training_batch(store,qids,cfg,i,device,hard)
            if amp:torch.cuda.synchronize()
            start=time.perf_counter()
            update=preflight_update(wrapped,model,optimizer,scaler,params,batch,device,amp,f'real_microbatch_{i+1}')
            retry_count+=update['overflow_retries']
            if amp:torch.cuda.synchronize()
            times.append(time.perf_counter()-start)
        # One discarded synthetic worst-shape step catches longer-document memory failures before training.
        optimizer.zero_grad(set_to_none=True)
        if cfg['kind']=='cross':
            n=batch['ids'].shape[0];T=cfg['pair_tokens'];batch['ids']=torch.randint(0,model.backbone.config.vocab_size,(n,T),device=device);batch['valid']=torch.ones(n,T,device=device,dtype=torch.bool)
            batch['segments']=torch.zeros(n,T,device=device,dtype=torch.long);batch['segments'][:,T//2:]=1
        else:
            B=batch['documents'];M=cfg['max_chunks'];T=cfg['chunk_tokens'];batch['ids']=torch.randint(0,model.backbone.config.vocab_size,(B*M,T),device=device);batch['valid']=torch.ones(B*M,T,device=device,dtype=torch.bool)
            batch['owner']=torch.arange(B,device=device).repeat_interleave(M);batch['slot']=torch.arange(M,device=device).repeat(B)
        stress_update=preflight_update(wrapped,model,optimizer,scaler,params,batch,device,amp,'worst_shape')
        retry_count+=stress_update['overflow_retries']
        checksum=torch.stack([p.detach().float().sum() for p in params]).sum()
        if world>1:
            lo=checksum.clone();hi=checksum.clone();dist.all_reduce(lo,op=dist.ReduceOp.MIN);dist.all_reduce(hi,op=dist.ReduceOp.MAX)
            if not torch.allclose(lo,hi,rtol=1e-6,atol=1e-5):raise ValueError('DDP rank parameters diverged.')
        result={'world_size':world,'finite_update':True,'micro_batch':cfg['micro_batch'],'chunk_tokens':cfg['chunk_tokens'],'max_chunks':cfg['max_chunks'],'seconds_per_microbatch':sum(times[1:])/2,
                'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30 if amp else None,'parameters':report,'worst_shape_update_passed':True,
                'amp_scale':float(scaler.get_scale()),'overflow_retries':retry_count,'last_finite_loss':stress_update['loss'],
                'source':'Timing uses real training microbatches. Peak includes one discarded synthetic step at configured maximum chunk/pair shapes.'}
        if rank_id==0:atomic_json(run/'preflight.json',result);print(json.dumps(result),flush=True)
        barrier(device)
        if dist.is_initialized():dist.destroy_process_group()
        return
    sampler=DistributedSampler(train,num_replicas=world,rank=rank_id,shuffle=True,seed=cfg['seed']);loader=DataLoader(train,batch_size=cfg['micro_batch'],sampler=sampler,num_workers=0,collate_fn=lambda x:x)
    total=max(1,math.ceil(len(loader)/cfg['accumulation'])*cfg['epochs']);updates=0;best=-1.;stale=0;start_epoch=0;best_epoch=0;elapsed_base=0;started=time.perf_counter();last=run/'last.pt';skipped=0;consecutive_skips=0
    if last.exists():
        saved=torch.load(last,map_location='cpu',weights_only=False)
        if saved['signature']!=signature:raise ValueError('Resume inputs/code changed. Use a new PROJECT.')
        model.load_state_dict(saved['model'],strict=True);optimizer.load_state_dict(saved['optimizer']);scaler.load_state_dict(saved['scaler'])
        start_epoch=saved['epoch'];updates=saved['updates'];best=saved['best'];stale=saved['stale'];best_epoch=saved['best_epoch'];elapsed_base=saved['elapsed_hours']
        r=saved['rng'][rank_id];random.setstate(r['python']);np.random.set_state(r['numpy']);torch.set_rng_state(r['torch'])
        if amp:torch.cuda.set_rng_state(r['cuda'],device)
    if stale>=cfg['patience']:start_epoch=cfg['epochs']
    if rank_id==0:print(json.dumps({'experiment':cfg['name'],'train_queries':len(train),'effective_query_batch':world*cfg['micro_batch']*cfg['accumulation'],'cross_rank_in_batch_negatives':world>1 and not model.cross,'parameters':report}),flush=True)
    probe=query_ids(store,'dev',cfg['dev_probe'],cfg['seed'])
    for epoch in range(start_epoch,cfg['epochs']):
        model.train();sampler.set_epoch(epoch);optimizer.zero_grad(set_to_none=True);running=0
        for i,qids in enumerate(loader):
            sync=(i+1)%cfg['accumulation']==0 or i+1==len(loader);group=(i//cfg['accumulation'])*cfg['accumulation'];divisor=min(cfg['accumulation'],len(loader)-group)
            # All ranks participate in candidate gathering even in no_sync microbatches.
            batch=training_batch(store,qids,cfg,epoch,device,hard,kd);ctx=wrapped.no_sync() if world>1 and not sync else contextlib.nullcontext()
            with ctx:
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):loss=wrapped(batch)
                if not all_finite(loss,device):raise FloatingPointError(f'Nonfinite forward loss at epoch {epoch+1}, batch {i+1}; not a loss-scaler-only overflow.')
                scaler.scale(loss/divisor).backward()
            running+=float(loss.detach())
            if sync:
                progress=updates/total;warm=.06;factor=min(1.,(updates+1)/max(1,total*warm)) if progress<warm else .1+.9*.5*(1+math.cos(math.pi*(progress-warm)/(1-warm)))
                for g in optimizer.param_groups:g['lr']=cfg['lr']*factor
                norm,stepped=finish_update(model,optimizer,scaler,params,device,f'epoch_{epoch+1}_batch_{i+1}')
                skipped+=not stepped;consecutive_skips=0 if stepped else consecutive_skips+1
                optimizer.zero_grad(set_to_none=True);updates+=int(stepped)
                if consecutive_skips>=12:raise FloatingPointError('Persistent gradient overflow during training; inspect the printed diagnostics.')
                if rank_id==0 and stepped and updates%cfg['log_every']==0:print(json.dumps({'epoch':epoch+1,'updates':updates,'loss':running/(i+1),'gradient_norm':float(norm),'amp_scale':float(scaler.get_scale()),'amp_skipped_updates':skipped,'elapsed_hours':elapsed_base+(time.perf_counter()-started)/3600}),flush=True)
        raw=rank(model,store,'dev',probe[rank_id::world],cfg,device,base_rankings(cfg,'dev'))
        atomic_json(run/f'dev_epoch_{epoch+1}.rank{rank_id}.json',raw);barrier(device);decision=[None]
        if rank_id==0:
            merged={}
            for r in range(world):merged.update(read_json(run/f'dev_epoch_{epoch+1}.rank{r}.json'))
            if set(merged)!=set(probe):raise ValueError('Incomplete validation probe rankings.')
            selection=select_fusion(store,merged);value=selection['metrics']['micro_f1_at7'];improved=value>best+cfg['min_delta']
            if improved:best=value;best_epoch=epoch+1;stale=0
            else:stale+=1
            decision[0]={'best':best,'best_epoch':best_epoch,'stale':stale,'improved':improved,'selection':selection};atomic_json(run/f'metrics_epoch_{epoch+1}.json',decision[0]);print(json.dumps({'epoch':epoch+1,**decision[0]}),flush=True)
        if world>1:dist.broadcast_object_list(decision,src=0,device=device)
        dec=decision[0];best=dec['best'];stale=dec['stale'];best_epoch=dec['best_epoch']
        state={'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state(device) if amp else None};rng=[None]*world
        if world>1:dist.all_gather_object(rng,state)
        else:rng=[state]
        if rank_id==0:
            saved={'model':model.state_dict(),'signature':signature,'lineage':lineage,'epoch':epoch+1,'best':best,'best_epoch':best_epoch,'stale':stale,'updates':updates,'rng':rng,'parameters':report,'elapsed_hours':elapsed_base+(time.perf_counter()-started)/3600}
            if dec['improved']:save(bestpath,saved);atomic_json(run/'best_metrics.json',dec)
            save(last,{**saved,'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict()})
        barrier(device)
        if stale>=cfg['patience']:break
    if dist.is_initialized():dist.destroy_process_group()
if __name__=='__main__':main()
'''
(CODE/'p7_run.py').write_text(module_source, encoding='utf-8')
print('Wrote p7_run.py with calibrated AMP overflow handling')

In [ ]:
# p7_workflow.py
module_source = r'''"""Kaggle orchestration: seven own-foundation experiments, cached outputs and optional fusion."""
import itertools,json,math,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
import numpy as np
from p7_data import FILES,Store,read_json,atomic_json,file_hash,digest
from p7_eval import rank,ranking_lists,predictions,select_fusion,metrics,pack

def find_phase1(explicit=None):
    if explicit:
        p=Path(explicit)
        if not (p/'tokenizer.json').is_file():raise FileNotFoundError(f'No tokenizer.json under {p}. Attach original Phase-1 outputs.')
        return p
    candidates=sorted({p.parent for p in Path('/kaggle/input').rglob('tokenizer.json') if p.parent.name=='capstone_phase1'})
    if len(candidates)!=1:raise ValueError('Set PHASE1_ROOT to your unique original capstone_phase1 directory.')
    return candidates[0]
def obtain_data(project,local_root=None,revision=None):
    names=sum([list(p) for p in FILES.values()],[])
    if not local_root:
        roots=sorted({p for p in Path('/kaggle/input').rglob('Phase_7_PCR') if all((p/n).is_file() for n in names)})
        if len(roots)==1:local_root=roots[0]
        elif len(roots)>1:raise ValueError('Multiple Phase-7 datasets. Set DATA_ROOT explicitly.')
    if local_root:
        root=Path(local_root)
        if (root/'Phase_7_PCR').is_dir():root/='Phase_7_PCR'
        missing=[n for n in names if not (root/n).is_file()]
        if missing:raise FileNotFoundError(f'Missing official Phase-7 files: {missing}')
        atomic_json(Path(project)/'data_source.json',{'kind':'attached_official_phase7','root':str(root),'sha256':{n:file_hash(root/n) for n in names}});return str(root)
    from huggingface_hub import HfApi,hf_hub_download
    token=os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:token=None
    repo='Exploration-Lab/CS-781-Capstone';root=Path(project)/'data';path=root/'source.json';old=read_json(path) if path.exists() else None
    try:
        commit=old['revision'] if old else HfApi(token=token).dataset_info(repo,revision=revision).sha
        if old and revision and revision!=commit:raise ValueError('Requested revision differs from cached official commit.')
        for name in names:
            target=root/name
            if target.is_file() and old:
                if old['sha256'].get(name)!=file_hash(target):raise ValueError('Cached official data changed. Choose a new PROJECT.')
                continue
            src=hf_hub_download(repo,'Phase_7_PCR/'+name,repo_type='dataset',revision=commit,token=token)
            target.parent.mkdir(parents=True,exist_ok=True)
            if target.exists():
                if file_hash(target)!=file_hash(src):raise ValueError('Partial downloaded data conflicts with the pinned dataset.')
            else:
                # Share immutable HF bytes on the same filesystem instead of duplicating large candidate pools.
                try:os.link(Path(src).resolve(),target)
                except OSError:shutil.copy2(src,target)
        atomic_json(path,{'repo':repo,'revision':commit,'sha256':{n:file_hash(root/n) for n in names}})
    except Exception as e:raise RuntimeError('Attach official Phase_7_PCR or enable Internet, accept dataset terms, and add an HF_TOKEN Kaggle secret.') from e
    return str(root)
def configurations(project,phase1,manifest,settings):
    project=Path(project);phase1=Path(phase1);cache=read_json(manifest)
    specs=[('01_decoder_full','decoder','full','dense',settings['full_epochs'],5e-5),
           ('02_encoder_decoder_partial','encoder_decoder','partial','dense',settings['peft_epochs'],1e-4),
           ('03_local_gqa_lorma','local_gqa','lorma','late',settings['peft_epochs'],2e-4),
           ('04_hybrid_prompt','hybrid','prompt','dense',settings['peft_epochs'],2e-3),
           ('05_decoder_ia3','decoder','ia3','dense',settings['peft_epochs'],1e-3),
           ('06_decoder_cross_encoder','decoder','full','cross',settings['cross_epochs'],3e-5),
           ('07_decoder_student_kd','decoder','kd','dense',settings['student_epochs'],5e-5)]
    result=[]
    for name,arch,method,kind,epochs,lr in specs:
        parent=phase1/'runs'/('main_'+arch)/'best_model.pt'
        if not parent.is_file():raise FileNotFoundError(f'Missing original parent {parent}')
        cfg={'name':name,'architecture':arch,'method':method,'kind':kind,'parent':str(parent),'manifest':str(manifest),'run':str(project/'runs'/name),
             'seed':settings['seed'],'epochs':epochs,'lr':lr,'patience':settings['patience'],'min_delta':.0001,
             'micro_batch':settings['micro_batch'],'accumulation':settings['accumulation'],'negatives':settings['negatives'],
             'chunk_tokens':settings['chunk_tokens'],'max_chunks':settings['max_chunks'],'chunk_microbatch':settings['chunk_microbatch'],
             'pair_tokens':settings['pair_tokens'],'pair_microbatch':settings['pair_microbatch'],'index_batch':settings['index_batch'],
             'inference_pairs':settings['inference_pairs'],'rerank_topk':settings['rerank_topk'],'dev_probe':settings['dev_probe'],
             'teacher_candidates':settings['teacher_candidates'],'temperature':.05,'kd_temperature':2.,'kd_weight':.5,
             'rank':16,'alpha':32,'unfrozen_layers':2,'embedding_dim':256,'prompt_tokens':32 if method=='prompt' else 0,
             'checkpointing':True,'dropout':.05,'bos':cache['special']['bos'],'log_every':25,
             'hard_file':str(project/'hard_negatives.json') if settings['mine_hard_negatives'] and name!='01_decoder_full' else None,
             'base_run':str(project/'runs/01_decoder_full') if kind=='cross' else None,'kd_file':str(project/'teacher_scores.json')}
        path=project/'configs'/f'{name}.json'
        if path.exists() and read_json(path)!=cfg:raise ValueError(f'{name} configuration changed; choose a new PROJECT.')
        atomic_json(path,cfg);result.append(str(path))
    return result
def launch(config,action='train',world=2,**kwargs):
    module=Path(__file__).parent/'p7_run.py';command=[sys.executable,'-m','torch.distributed.run','--standalone',f'--nproc_per_node={world}',str(module),'--config',str(config),'--action',action]
    for k,v in kwargs.items():command+=['--'+k.replace('_','-'),str(v)]
    env=os.environ.copy();env.update(TOKENIZERS_PARALLELISM='false',OMP_NUM_THREADS='2',PYTHONUNBUFFERED='1',PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True');env['PYTHONPATH']=str(module.parent)+os.pathsep+env.get('PYTHONPATH','')
    cfg=read_json(config);path=Path(cfg['run'])/(action+'.log');path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('a',encoding='utf-8') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in process.stdout:print(line,end='');log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError(f'{cfg["name"]} {action} failed; inspect {path}. Completed submissions remain saved.')
def forecast(configs,world=2):
    rows=[];total=0
    for path in configs:
        cfg=read_json(path);pre=read_json(Path(cfg['run'])/'preflight.json');store=Store(cfg['manifest']);n=sum(bool(r['relevant']) for r in store.splits['train']['queries'].values())
        batches=math.ceil(math.ceil(n/world)/cfg['micro_batch']);hours=batches*cfg['epochs']*pre['seconds_per_microbatch']/3600;total+=hours
        rows.append({'name':cfg['name'],'training_hours_at_max_epochs':hours,'seconds_per_microbatch':pre['seconds_per_microbatch'],'peak_preflight_gib':pre['peak_reserved_gib']})
    print(json.dumps(rows,indent=2));print(f'Measured training-only forecast: {total:.2f} hours. Additional preprocessing, validation/indexing, mining, distillation-target scoring, final inference and I/O are excluded. Example lengths vary; this is not a runtime guarantee.')
    return rows
def final_rankings(config,store,world):
    cfg=read_json(config);run=Path(cfg['run']);raws={}
    for split in ('dev','test'):
        path=run/f'{split}_raw.json';launch(config,'rank',world,split=split,output=path);raws[split]=read_json(path)
    selection=select_fusion(store,raws['dev']);weight=selection['neural_weight'];atomic_json(run/'full_dev_selection.json',selection)
    for split in ('dev','test'):
        lists=ranking_lists(raws[split],weight);atomic_json(run/f'{split}_rankings.json',lists);atomic_json(run/f'{split}_prediction.json',{q:r[:7] for q,r in lists.items()})
    return selection
def mine(config,store,project,world):
    cfg=read_json(config);run=Path(cfg['run']);path=run/'train_raw.json';launch(config,'rank',world,split='train',output=path)
    weight=read_json(run/'full_dev_selection.json')['neural_weight'];lists=ranking_lists(read_json(path),weight);atomic_json(run/'train_rankings.json',lists)
    # Known positives are filtered by the sampler; mining cannot use development/test relevance.
    atomic_json(Path(project)/'hard_negatives.json',{'data_fingerprint':store.manifest['fingerprint'],'source_checkpoint_sha256':file_hash(run/'best.pt'),'rankings':lists})
def make_teacher(config,store,project,world):
    cfg=read_json(config);project=Path(project);checkpoint=Path(cfg['run'])/'best.pt';path=project/'teacher_scores.json'
    identity={'data_fingerprint':store.manifest['fingerprint'],'teacher_checkpoint':str(checkpoint),'teacher_checkpoint_sha256':file_hash(checkpoint),
              'fusion_selection_sha256':file_hash(Path(cfg['run'])/'full_dev_selection.json')}
    if path.exists():
        old=read_json(path)
        if any(old.get(k)!=v for k,v in identity.items()):raise ValueError('Teacher cache inputs changed. Use a new PROJECT.')
        return
    scores=project/'teacher_train_raw.json';launch(config,'teach',world,output=scores)
    values=read_json(scores);allowed=store.splits['train']['queries']
    if not set(values)<=set(allowed):raise ValueError('Held-out queries entered the teacher cache.')
    if any(set(v['ids'])-set(store.splits['train']['candidates']) for v in values.values()):raise ValueError('Teacher candidates outside train pool.')
    atomic_json(path,{**identity,'scores':values,'objective':'Cached actual retrieval logits on train-only gold+shortlist candidate groups. Student uses supervised multi-positive loss plus KL distillation.'})
def make_ensemble(project,results,store):
    # Optional score candidate, not an additional trained architecture or mandatory-method experiment.
    ranked=sorted(results,key=lambda x:x['selection']['metrics']['micro_f1_at7'],reverse=True);candidates=[]
    for n in (2,3):
        chosen=ranked[:n]
        if len(chosen)<n:continue
        for primary_weight in (.5,1.,2.):
            weights=[primary_weight]+[1.]*(n-1);preds={}
            for split in ('dev','test'):
                rankings=[read_json(Path(project)/'runs'/r['name']/f'{split}_rankings.json') for r in chosen];pred={}
                for q in store.splits[split]['queries']:
                    scores={}
                    for weight,lists in zip(weights,rankings):
                        for i,c in enumerate(lists[q]):scores[c]=scores.get(c,0.)+weight/(60+i+1)
                    pred[q]=sorted(scores,key=lambda c:(-scores[c],c))[:7]
                preds[split]=pred
            m=metrics(store,'dev',preds['dev']);candidates.append({'members':[r['name'] for r in chosen],'weights':weights,'metrics':m,'prediction':preds['test']})
    if not candidates:return None
    best=max(candidates,key=lambda r:r['metrics']['micro_f1_at7']);prediction=best.pop('prediction');item=pack(project,'ensemble_rrf',prediction,store)
    atomic_json(Path(project)/'ensemble_selection.json',{**best,'candidate_metrics':[{k:v for k,v in r.items() if k!='prediction'} for r in candidates]})
    return {**item,'selection':{'metrics':best['metrics']},'kind':'ensemble','method':'rank_fusion','architecture':'multiple'}
def run_all(project,configs,settings,world=2):
    project=Path(project);store=Store(read_json(configs[0])['manifest']);results=[];started=time.perf_counter()
    for path in configs:
        cfg=read_json(path);run=Path(cfg['run']);completed=run/'completed.json'
        identity=run_identity(cfg,store,world)
        if completed.exists():
            old=read_json(completed);item=old['result']
            if old['identity']!=identity or old['checkpoint_sha256']!=file_hash(run/'best.pt'):raise ValueError('Completed run changed. Choose a new PROJECT.')
            if not Path(item['submission_zip']).is_file():raise FileNotFoundError('Completed submission ZIP missing.')
            results.append(item);print('Reusing completed submission:',cfg['name']);continue
        if cfg['method']=='kd':
            eligible=[p for p in configs if read_json(p)['method']!='kd' and (Path(read_json(p)['run'])/'full_dev_selection.json').is_file()]
            teacher=max(eligible,key=lambda p:read_json(Path(read_json(p)['run'])/'full_dev_selection.json')['metrics']['micro_f1_at7']);make_teacher(teacher,store,project,world)
        launch(path,'train',world);selection=final_rankings(path,store,world)
        item={**pack(project,cfg['name'],read_json(run/'test_prediction.json'),store),'architecture':cfg['architecture'],'kind':cfg['kind'],'method':cfg['method'],'selection':selection,'best_epoch':read_json(run/'best_metrics.json')['best_epoch']}
        if cfg['name']=='01_decoder_full':
            # The cross-encoder needs split-specific first-stage rankings; training rankings also enable one-time mining.
            mine(path,store,project,world)
        atomic_json(completed,{'identity':run_identity(cfg,store,world),'checkpoint_sha256':file_hash(run/'best.pt'),'result':item});results.append(item);atomic_json(project/'results.json',results)
        print(json.dumps({'completed':cfg['name'],'dev_metrics':selection['metrics'],'elapsed_all_hours':(time.perf_counter()-started)/3600}),flush=True)
    atomic_json(project/'results.json',results);ranked=sorted(results,key=lambda r:r['selection']['metrics']['micro_f1_at7'],reverse=True)
    if settings.get('ensemble',True):
        ensemble=make_ensemble(project,results,store)
        if ensemble:ranked=sorted(ranked+[ensemble],key=lambda r:r['selection']['metrics']['micro_f1_at7'],reverse=True)
    atomic_json(project/'leaderboard.json',ranked)
    if ranked:shutil.copy2(ranked[0]['submission_zip'],project/'best_submission.zip')
    distinct=len({r['prediction_sha256'] for r in results});compliance={'trained_models':len(results),'distinct_model_prediction_files':distinct,'methods':sorted({r['method'] for r in results}),'families':sorted({r['architecture'] for r in results}),'meets_six_distinct_submission_count':distinct>=6}
    atomic_json(project/'compliance.json',compliance)
    if distinct<len(results):print('WARNING: Some trained models have identical test rankings. Repeated predictions do not count as distinct submissions; inspect compliance.json.')
    return ranked
def run_identity(cfg,store,world):
    depends={}
    for key in ['hard_file']+(['kd_file'] if cfg['method']=='kd' else []):
        if cfg.get(key):depends[key]=file_hash(cfg[key]) if Path(cfg[key]).is_file() else None
    if cfg.get('base_run'):
        depends['base_run']={s:file_hash(Path(cfg['base_run'])/f'{s}_rankings.json') for s in ('train','dev','test') if (Path(cfg['base_run'])/f'{s}_rankings.json').is_file()}
    return {'config_sha256':digest(cfg),'data_fingerprint':store.manifest['fingerprint'],'parent_sha256':file_hash(cfg['parent']),
            'code_sha256':{p.name:file_hash(p) for p in Path(__file__).parent.glob('*.py')},'world_size':world,'dependencies':depends}
def archive_all(project):
    project=Path(project);output=project.parent/(project.name+'_all_outputs.zip');size=sum(p.stat().st_size for p in project.rglob('*') if p.is_file())
    if shutil.disk_usage(project.parent).free<size+512*1024**2:raise OSError('Insufficient free disk for the all-output ZIP. Individual submissions and checkpoints remain saved.')
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in sorted(project.rglob('*')):
            if p.is_file() and '__pycache__' not in p.parts and not p.name.endswith('.tmp'):z.write(p,str(p.relative_to(project.parent)))
    return output
'''
(CODE/'p7_workflow.py').write_text(module_source, encoding='utf-8')
print('Wrote p7_workflow.py')

In [ ]:
import importlib, platform
sys.path.insert(0, str(CODE)); importlib.invalidate_caches()
from p7_data import prepare_cache, Store, read_json, atomic_json, file_hash
from p7_workflow import find_phase1, obtain_data, configurations, launch, forecast, run_all, archive_all
PHASE1_ROOT = find_phase1(PHASE1_ROOT)
TOKENIZER_PATH = PHASE1_ROOT/'tokenizer.json'
data_root = obtain_data(PROJECT, DATA_ROOT, DATA_REVISION)
MANIFEST = prepare_cache(data_root, TOKENIZER_PATH, PROJECT/'token_cache', SETTINGS)
store = Store(MANIFEST)
print(json.dumps(store.manifest['audit'], indent=2))
CONFIGS = configurations(PROJECT, PHASE1_ROOT, MANIFEST, SETTINGS)
for p in CONFIGS:
    cfg = read_json(p); print(cfg['name'], cfg['architecture'], cfg['method'], cfg['kind'])
atomic_json(PROJECT/'environment.json', {'python': platform.python_version(), 'torch': torch.__version__,
    'gpu': [torch.cuda.get_device_name(i) for i in range(WORLD_SIZE)], 'settings': SETTINGS,
    'tokenizer_sha256': file_hash(TOKENIZER_PATH), 'data_fingerprint': store.manifest['fingerprint']})
with (PROJECT/'requirements_executed.txt').open('w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)


## Inspect shortlist coverage before training

The audit reports BM25 Micro-F1@7, recall of gold cases inside the top-128 lexical shortlist, and a Micro-F1@7 upper bound achievable inside that shortlist. The cross-encoder further narrows to a lexical/neural top-64 union. **No reranker can recover cases outside its shortlist.** If coverage is inadequate, increase `lexical_topk` and/or `rerank_topk` before training and use a new `PROJECT`. Larger values increase indexing/reranking time; no score-neutral shortcut is claimed.

The primary metric is **Micro-F1@7**, with precision/recall as secondary metrics. The evaluator compares seven predictions with the **complete** gold set, including queries with fewer or more than seven relevant cases:

$$F1@7=\frac{2TP}{2TP+FP+FN}.$$

The data loader never pads/truncates gold sets to seven. Evaluation always requires exactly seven unique candidate IDs from the matching split pool.

In [ ]:
print('BM25 baseline:', json.dumps(store.manifest['audit']['baseline_bm25'], indent=2))
print('Shortlist ceilings:', json.dumps(store.manifest['audit']['shortlist_ceilings'], indent=2))
# Reference files for exact local evaluation; test truth is never created.
atomic_json(PROJECT/'dev_truth.json', {q: r['relevant'] for q,r in store.splits['dev']['queries'].items()})
atomic_json(PROJECT/'dev_candidate_ids.json', sorted(store.splits['dev']['candidates']))


## Actual two-GPU preflight and measured training forecast

Each preflight launches two ranks on different real query examples, performs finite optimizer updates, checks matching rank parameter checksums, and adds one synthetic maximum-shape update to expose memory failures before the long run. Its modified weights are discarded. It runs the smaller student on supervised data before teacher scores exist.

The forecast uses measured real-example training steps. **It excludes tokenization/BM25 construction, validation re-indexing, mining, teacher scoring, final inference and I/O.** Validation and cross-encoder inference can be substantial. Peak memory includes the maximum-shape test; untested backend behavior remains possible until this succeeds on Kaggle.

In [ ]:
for config_path in CONFIGS:
    launch(config_path, action='preflight', world=WORLD_SIZE)
    print('PASS actual two-GPU preflight:', read_json(config_path)['name'])
training_forecast = forecast(CONFIGS, WORLD_SIZE)
atomic_json(PROJECT/'training_forecast.json', training_forecast)


## Train all seven and save each candidate

Each epoch evaluates a fixed, seeded development probe (up to 128 queries) and selects by its Micro-F1@7. The best checkpoint is then evaluated on all official development queries; the fusion weight is selected there. These are **development selection results, not independent test estimates**. No official test relevance or leaderboard feedback enters tuning.

Run 01's final rankings are reused to mine training negatives once and to help build the cross-encoder shortlist. After run 06, the strongest completed teacher creates cached training relevance scores. Run 07 learns supervised relevance plus KL distillation in a smaller model. Epoch counts are ceilings; patience=2 may stop earlier. There is no unvalidated train+dev refit.

`last.pt` resumes at the last completed epoch with optimizer/scaler/RNG state. Rerunning the cell reuses completed submissions. For another Kaggle session, restore the entire project directory to `/kaggle/working` with unchanged paths/code/configuration before running these cells. Read-only `/kaggle/input` is for inputs, not writable output.

In [ ]:
ranked = run_all(PROJECT, CONFIGS, SETTINGS, world=WORLD_SIZE)
print(json.dumps(ranked, indent=2))
print('Submission audit:', json.dumps(read_json(PROJECT/'compliance.json'), indent=2))


## Official evaluator cross-check

The unchanged evaluator checks a trained candidate's development predictions, query coverage, unique IDs, candidate-pool membership and exact Micro-F1@7. The optional ensemble has its development score in `ensemble_selection.json`; the best overall test ZIP is already copied to `best_submission.zip`.

In [ ]:
best_model = sorted(read_json(PROJECT/'results.json'), key=lambda r:r['selection']['metrics']['micro_f1_at7'], reverse=True)[0]
subprocess.check_call([sys.executable, str(CODE/'evaluate_phase7_official.py'),
    '--truth', str(PROJECT/'dev_truth.json'),
    '--candidate-ids', str(PROJECT/'dev_candidate_ids.json'),
    '--prediction', str(PROJECT/'runs'/best_model['name']/'dev_prediction.json'),
    '--output', str(PROJECT/'official_dev_score.json')])
print('Best overall selected candidate:', read_json(PROJECT/'leaderboard.json')[0]['name'])
print('Best submission:', PROJECT/'best_submission.zip')


## Download submissions and backup

Each model ZIP contains **only `prediction.json` at its root**. The JSON maps each original test query ID to exactly seven unique, ranked candidate IDs from `test_candidates.json`. `best_submission.zip` is a copy of the development-selected best model/ensemble ZIP.

The seven model ZIPs are genuine adaptation experiments. Identical rankings are detected; repeated predictions do not satisfy the distinct-submission requirement, even if their model weights differ. The ensemble is an additional score candidate and is not counted as another trained method.

The all-outputs backup ZIP includes code, indices, token cache, audits, logs, teacher scores, checkpoints, validation outputs and all submissions. It is **not** a Codabench submission. The backup is outside the project directory, so it cannot include itself. Disk availability is checked before creating it.

In [ ]:
from IPython.display import display, FileLink
for item in read_json(PROJECT/'results.json'):
    display(FileLink(item['submission_zip']))
if (PROJECT/'submissions/ensemble_rrf.zip').is_file():
    display(FileLink(str(PROJECT/'submissions/ensemble_rrf.zip')))
display(FileLink(str(PROJECT/'best_submission.zip')))
ALL_OUTPUTS_ZIP = archive_all(PROJECT)
print('Backup GiB:', round(ALL_OUTPUTS_ZIP.stat().st_size/2**30, 2))
display(FileLink(str(ALL_OUTPUTS_ZIP)))
